## Configurazione e pacchetti

In [ ]:
# ==============================================================================
#  BENCHMARK TRASCRITTOMICA SPAZIALE — pipeline_v7
#  File unico, da far girare nel notebook. Per ogni dataset si cambia solo il
#  BLOCCO DATASET (percorsi, centro, regione di origine di SPATA2).
#
#  METODI CONFRONTATI
#    A) GLM_linear / GLM_gaussian sulla distanza dal centro (barcode)
#    D) gli stessi GLM con PSEUDOTIME (Gaston, SpaceFlow, Slingshot, Monocle)
#    E) riferimenti 2D center-free: SpatialDE originale (da CSV), Moran's I,
#       Laplacian Score
#    F) Hybrid_GLM (lineare vs gaussiano, selezione BIC)
#    G) centro inferito dai dati (9c) e GLM rifatti da li' (9d, *_inferred)
#    H) tool esterno: SPATA2, workflow dell'articolo (11b)
# ==============================================================================

suppressPackageStartupMessages({
  library(SpatialExperiment)
  library(SingleCellExperiment)
  library(S4Vectors)
  library(Matrix)
  library(readr)
  library(dplyr)
  library(tidyr)
  library(scuttle)
  library(scater)
  library(rtracklayer)
  library(ggplot2)
  library(doParallel)
  library(foreach)
  library(edgeR)
  library(msigdbr)
  library(clusterProfiler)
  library(FNN)
})




# ##############################################################################
# ##                                                                          ##
# ##   BLOCCO DATASET — L'UNICA PARTE DA SOSTITUIRE PER OGNI DATASET          ##
# ##   (da qui fino a "FINE BLOCCO DATASET"). Tutto il resto del file e'      ##
# ##   comune ai 6 dataset e non va toccato.                                  ##
# ##                                                                          ##
# ##############################################################################
S <- "/home/ammendola/laboratorio/spatial"
DATA_SOURCE          <- "visium"
INPUT_PATH           <- file.path(S, "topo/data/SpatialTranscriptomics/samples/heme_1000/outs")
VISIUM_TYPE          <- "HDF5"
POSITIONS_PATH       <- file.path(S, "topo/data/SpatialTranscriptomics/samples/heme_1000/outs/spatial/tissue_positions_list.csv")
RDATA_PATH           <- NULL

GTF_PATH             <- "/home/ammendola/laboratorio/reference/mouse/Mus_musculus.GRCm39.115.gtf"
GENE_ID_TYPE         <- "ensembl"

ANATOMY_PATH         <- file.path(S, "topo/data/SpatialTranscriptomics/samples/heme_1000/heme_1000_anatomy.csv")

SLINGSHOT_PATH       <- file.path(S, "topo/data/pseudotime_slingshot.csv")
MONOCLE_PATH         <- file.path(S, "topo/data/pseudotime_monocle3.csv")
SPACEFLOW_PATH       <- file.path(S, "topo/data/spaceflow_output/spot_pseudotime.csv")
GASTON_PATH          <- file.path(S, "topo/data/gaston_output/gaston_isodepth.csv")
SPATIALDE_PATH       <- file.path(S, "topo/data/spatialde/spatial_de_results.csv")

GOLDEN_STANDARD_PATH <- file.path(S, "topo/data/paper_genes.csv")
GS_ALPHA             <- 0.05
ENRICHMENT_CONTEXT   <- "inflammation"
ORGANISM             <- "Mus musculus"
SAMPLE_ID            <- "heme_1000"
OUTPUT_DIR           <- file.path("/home/ammendola/laboratorio/spatial/topo/code/benchmark_v7", SAMPLE_ID)

ORIGINE_CENTRO <- list(tipo = "csv", colonna = "injection_site", valore = "heme",
                       path = file.path(S, "topo/data/SpatialTranscriptomics/samples/heme_1000/heme_1000_injection_site.csv"))

SPATA2_VISIUM_DIR    <- file.path(S, "topo/data/SpatialTranscriptomics/samples/heme_1000/outs")
SPATA2_ORIGIN        <- list(tipo = "csv", colonna = "injection_site", valore = "heme",
                             path = file.path(S, "topo/data/SpatialTranscriptomics/samples/heme_1000/heme_1000_injection_site.csv"))
SPATA2_PARAMETRI_PAPER <- FALSE
# ##############################################################################
# ##   FINE BLOCCO DATASET                                                    ##
# ##############################################################################

# ===== CODICE COMUNE (identico nei sei notebook) =====
# --- centro: medoide della regione di origine, tutto in pixel -----------------
# Regola unica per i sei dataset, fissata prima di guardare le AUC.
# Cambia solo ORIGINE_CENTRO (sopra); il resto e' identico nei sei notebook.

# Barcode degli spot della regione di origine, letti dalla fonte degli autori.
#   csv         : file con una colonna di annotazione (colonna == valore)
#   seurat      : oggetto Seurat degli autori (meta.data[[colonna]] == valore)
#   cluster_csv : cluster ricalcolati (seurat_cluster == valore)
#   img         : poligono disegnato dagli autori (image annotation in SPATA2)
origine_barcode <- function(o, xy) {
  if (o$tipo == "csv") {
    a <- read.csv(o$path, check.names = FALSE)
    return(a$Barcode[!is.na(a[[o$colonna]]) & a[[o$colonna]] == o$valore])
  }
  if (o$tipo == "seurat") {
    e <- new.env(); nm <- load(o$path, envir = e); md <- e[[nm[1]]]@meta.data
    bc <- rownames(md)[!is.na(md[[o$colonna]]) & md[[o$colonna]] == o$valore]
    return(sub("-[0-9]+$", o$suffisso, bc))
  }
  if (o$tipo == "cluster_csv") {
    a <- read.csv(o$path)
    return(a$barcode[as.character(a$seurat_cluster) == o$valore])
  }
  # img: il poligono e' in pixel dell'immagine lowres -> si riporta in pixel full-res
  sf <- jsonlite::fromJSON(file.path(dirname(POSITIONS_PATH), "scalefactors_json.json"))
  e <- new.env(); utils::data("image_annotations", package = "SPATA2", envir = e)
  poli <- as.data.frame(e$image_annotations[[o$campione]][[o$ids]]@area$outer)
  vx <- poli$x / sf$tissue_lowres_scalef; vy <- poli$y / sf$tissue_lowres_scalef
  # punto nel poligono (ray casting): dentro se attraversa un numero dispari di lati
  dentro <- rep(FALSE, nrow(xy)); j <- length(vx)
  for (i in seq_along(vx)) {
    attraversa <- ((vy[i] > xy$y) != (vy[j] > xy$y)) &
      (xy$x < (vx[j] - vx[i]) * (xy$y - vy[i]) / (vy[j] - vy[i]) + vx[i])
    dentro <- xor(dentro, attraversa); j <- i
  }
  rownames(xy)[dentro]
}

# Medoide della regione.
#   bc : barcode della regione di origine
#   xy : TUTTI gli spot in tessuto (rownames = barcode, colonne x, y in pixel)
medoide <- function(bc, xy) {
  bc <- intersect(bc, rownames(xy))
  xy <- as.matrix(xy[, c("x", "y")])

  # 1. Passo della griglia: mediana, su tutti gli spot in tessuto, della distanza
  #    dallo spot piu' vicino (100 um nel Visium, qui in pixel).
  d_tutti <- as.matrix(dist(xy)); diag(d_tutti) <- Inf
  passo <- median(apply(d_tutti, 1, min))

  # 2. Spot "vicini" = a meno di 1.5 passi (solo i primi vicini, catena di vicini).
  #    La regione puo' essere fatta di piu' pezzi staccati: ogni pezzo ha un numero.
  regione <- xy[bc, , drop = FALSE]
  pezzo <- if (nrow(regione) > 1) cutree(hclust(dist(regione), "single"), h = 1.5 * passo) else 1

  # 3. Si tiene solo il pezzo con piu' spot.
  pezzo_grande <- as.integer(names(which.max(table(pezzo))))
  regione <- regione[pezzo == pezzo_grande, , drop = FALSE]

  # 4. Per ogni spot: somma delle distanze da tutti gli altri spot del pezzo.
  somma_distanze <- rowSums(as.matrix(dist(regione)))

  # 5. Per ogni spot: distanza dal baricentro del pezzo (serve solo per i pari merito).
  dal_baricentro <- sqrt(rowSums(sweep(regione, 2, colMeans(regione))^2))

  # 6. Il medoide e' lo spot con la somma minima; a parita' (entro 1e-6) vince
  #    quello piu' vicino al baricentro.
  rownames(regione)[order(round(somma_distanze, 6), dal_baricentro)[1]]
}

# Spot in tessuto con le coordinate in pixel full-res (pxl_col = x, pxl_row = y).
pos <- read.csv(POSITIONS_PATH, header = FALSE)
if (!is.numeric(pos[[2]])) pos <- read.csv(POSITIONS_PATH)
pos <- pos[pos[[2]] == 1, ]
xy  <- data.frame(x = pos[[6]], y = pos[[5]], row.names = pos[[1]])

CENTER_BARCODE <- medoide(origine_barcode(ORIGINE_CENTRO, xy), xy)
cat("CENTER_BARCODE (medoide):", CENTER_BARCODE, "\n")
rm(pos, xy)


In [ ]:
# ##############################################################################
# ##   CONFIGURAZIONE COMUNE - identica per i 6 dataset                       ##
# ##############################################################################
 
PLOT_DIR             <- file.path(OUTPUT_DIR, "plots")
 
# --- SPATA2 (sezione 11b) --------------------------------------------------------
RUN_SPATA2           <- TRUE
# default del tool e raccomandazione del paper: i geni zero-inflated non vengono
# testati; qui finiscono in coda alla classifica (ordine casuale, seed fisso)
SPATA2_RM_ZERO_INFL  <- TRUE
SPATA2_SEED_CODA     <- 20260929
 
# distance e resolution: si accendono e spengono dal blocco dataset con
# SPATA2_PARAMETRI_PAPER; qui ci sono solo i valori
SPATA2_ARGS_PAPER    <- list(distance = "1.5mm", resolution = "50um")
if (!exists("SPATA2_PARAMETRI_PAPER") || !is.logical(SPATA2_PARAMETRI_PAPER))
  stop("Nel blocco dataset manca SPATA2_PARAMETRI_PAPER <- TRUE (1.5mm / 50um) o FALSE (default del tool).")
SPATA2_ARGS          <- if (SPATA2_PARAMETRI_PAPER) SPATA2_ARGS_PAPER else list()
 
# screening in parallelo (sezione 11b.3)
SPATA2_N_WORKERS          <- 40    # processi (limitati ai core concessi da taskset)
SPATA2_BLOCCHI_PER_WORKER <- 3     # blocchi di geni per processo: piu' blocchi = carico piu' bilanciato
SPATA2_PRINT_OGNI         <- 120   # secondi: stato stampato anche se nessun blocco finisce
 
# --- parametri generali --------------------------------------------------------
MIN_SPOTS_PER_GENE   <- 3
ALPHA_FDR            <- 0.05
N_CORES              <- 40
 
# --- Hybrid --------------------------------------------------------------------
RUN_HYBRID           <- TRUE   # Hybrid_GLM: lineare vs gaussiano, selezione BIC
 
RUN_GLM_PSEUDOTIME   <- TRUE
GLM_PT_MIN_SPOTS     <- 30
 
# --- riferimenti 2D center-free: Moran's I e Laplacian Score (sezione 10d) -----
# Input: logcounts (scuttle::logNormCounts), come squidpy, Seurat e Voyager.
# Pesi del grafo per Moran's I:
#   "W" : standardizzati per riga (default di spdep e di squidpy spatial_autocorr)
#   "B" : binari (come nella v3)
# Il Laplacian Score usa sempre il grafo binario simmetrico: la sua definizione
# richiede W simmetrica e il grado come misura sui nodi.
REF2D_MORAN_STYLE    <- "W"
 
# Controllo numerico di Moran's I contro spdep::moran.test su 5 geni (serve
# il pacchetto spdep; se manca il controllo viene saltato, non il calcolo).
REF2D_CHECK_SPDEP    <- TRUE
 
# --- SpatialDE originale (sezione 10b) -----------------------------------------
# Geni dell'universo di questo script assenti dal CSV di SpatialDE (il notebook
# Python filtra in modo diverso):
#   "coda"    : in fondo alla classifica, in ordine casuale, fdr = NA. Stesso
#               trattamento di SPATA2: AUC sullo stesso universo degli altri.
#   "escludi" : nessuna riga (comportamento della v3: AUC su un universo diverso).
SDE_ORIG_NON_TESTATI <- "coda"
SDE_ORIG_SEED_CODA   <- 20260929
 
# --- diagnostiche per leggere il benchmark (sezione 10f) ----------------------
# 10f: nullo spaziale dei GLM radiali (usa Z1_scan della 9c, nessun fit nuovo)
RUN_SPATIAL_NULL     <- TRUE
 
# core concessi al processo: con "taskset -c 0-19" sono 20, anche se la
# macchina ne ha di piu'. detectCores() vede tutta la macchina e non va bene qui.
N_CORES_DISPONIBILI <- tryCatch(length(parallel::mcaffinity()), error = function(e) 0L)
if (N_CORES_DISPONIBILI == 0L) N_CORES_DISPONIBILI <- parallel::detectCores()
N_CORES <- min(N_CORES, 40L, N_CORES_DISPONIBILI)
cat("Core disponibili (taskset):", N_CORES_DISPONIBILI, " | N_CORES effettivi (cap 40):", N_CORES, "\n")

## 0. AMBIENTE - TETTO DI PROCESSORI E FUNZIONI dplyr

In [ ]:
# ##############################################################################
# ##   0. AMBIENTE - TETTO DI PROCESSORI E FUNZIONI dplyr                     ##
# ##                                                                          ##
# ##   Il tetto di 40 va imposto a TUTTI i livelli di parallelismo, non solo  ##
# ##   ai worker espliciti. Le librerie di algebra lineare (BLAS: OpenBLAS o  ##
# ##   MKL) di default usano tutti i core della macchina a ogni prodotto di   ##
# ##   matrici; se poi si aprono N worker FORK, ognuno eredita quel BLAS e i  ##
# ##   thread totali diventano N x (core della macchina).                    ##
# ##                                                                          ##
# ##   Regola usata in tutto il file:                                        ##
# ##     - processo principale: BLAS a N_CORES thread                         ##
# ##     - prima di aprire nw worker: BLAS a floor(N_CORES / nw) thread,      ##
# ##       cosi' nw x thread_per_worker <= N_CORES; poi si torna a N_CORES   ##
# ##   Per controllare il BLAS da dentro R serve il pacchetto RhpcBLASctl:   ##
# ##   se manca, install.packages("RhpcBLASctl").                            ##
# ##############################################################################
 
# --- avanzamento: stampe con orario + file avanzamento.txt -------------------
# Ogni riga va nel notebook e nel file OUTPUT_DIR/avanzamento.txt. Il file si
# legge DURANTE la run, da un altro terminale:
#     tail -f <OUTPUT_DIR>/avanzamento.txt
dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
LOG_AVANZAMENTO <- file.path(OUTPUT_DIR, "avanzamento.txt")
T_INIZIO_RUN    <- Sys.time()
T_INIZIO_SEZ    <- Sys.time()
SEZ_CORRENTE    <- NA_character_
cat(format(Sys.time(), "%Y-%m-%d %H:%M:%S"), " INIZIO RUN ", SAMPLE_ID, "\n",
    sep = "", file = LOG_AVANZAMENTO)
 
minuti_da <- function(t) round(as.numeric(difftime(Sys.time(), t, units = "mins")), 1)
 
stampa <- function(..., sep = "") {
  testo <- paste(..., sep = sep)
  cat(format(Sys.time(), "%H:%M:%S"), " ", testo, "\n", sep = "")
  cat(format(Sys.time(), "%Y-%m-%d %H:%M:%S"), " ", testo, "\n",
      sep = "", file = LOG_AVANZAMENTO, append = TRUE)
  flush.console()
  invisible(NULL)
}
 
# intestazione di sezione: stessa riga "=== ... ===" di prima, piu' durata
# della sezione precedente e tempo dall'inizio della run
sezione_inizio <- function(..., sep = " ") {
  titolo <- trimws(paste(..., sep = sep))
  if (!is.na(SEZ_CORRENTE))
    stampa("fine sezione ", SEZ_CORRENTE, " (", minuti_da(T_INIZIO_SEZ), " min)")
  SEZ_CORRENTE <<- sub("\\.$", "", sub(" .*$", "", titolo))
  T_INIZIO_SEZ <<- Sys.time()
  cat("\n=== ", titolo, " ===\n", sep = "")
  stampa("INIZIO ", titolo, "  | run in corso da ", minuti_da(T_INIZIO_RUN), " min")
}
 
# stima del tempo residuo di un ciclo: fatti su totale, dal tempo t0
stima_residua <- function(fatti, totale, t0) {
  m <- minuti_da(t0)
  paste0(fatti, "/", totale, " (", round(100 * fatti / totale), "%) | trascorsi ", m,
         " min | stima residua ", round(m / max(fatti, 1) * (totale - fatti), 1), " min")
}
 
sezione_inizio("0. Ambiente")
stampa("Dataset: ", SAMPLE_ID, " | output: ", normalizePath(OUTPUT_DIR))
stampa("Avanzamento leggibile durante la run con:  tail -f ", normalizePath(LOG_AVANZAMENTO))
stampa("SPATA2: ", if (length(SPATA2_ARGS) == 0) "parametri di default" else
         paste(names(SPATA2_ARGS), unlist(SPATA2_ARGS), sep = " = ", collapse = ", "),
       " | rm_zero_infl = ", SPATA2_RM_ZERO_INFL, " | processi screening: ", SPATA2_N_WORKERS)
 
# --- variabili d'ambiente (lette da OpenMP/BLAS all'avvio dei processi) -----
Sys.setenv(OMP_NUM_THREADS        = as.character(N_CORES),
           OPENBLAS_NUM_THREADS   = as.character(N_CORES),
           MKL_NUM_THREADS        = as.character(N_CORES),
           VECLIB_MAXIMUM_THREADS = as.character(N_CORES))
 
# --- mclapply e BiocParallel (usato da fgsea dentro la GSEA) ----------------
options(mc.cores = N_CORES)
if (requireNamespace("BiocParallel", quietly = TRUE))
  BiocParallel::register(BiocParallel::MulticoreParam(workers = N_CORES), default = TRUE)
 
# --- funzione unica per fissare i thread del BLAS ---------------------------
HAS_BLASCTL <- requireNamespace("RhpcBLASctl", quietly = TRUE)
 
set_blas_threads <- function(n) {
  n <- max(1L, as.integer(n))
  if (HAS_BLASCTL) {
    RhpcBLASctl::blas_set_num_threads(n)
    RhpcBLASctl::omp_set_num_threads(n)
  }
  invisible(n)
}
 
set_blas_threads(N_CORES)
if (HAS_BLASCTL) {
  cat("BLAS: thread fissati a", N_CORES, "(RhpcBLASctl)\n")
} else {
  cat("ATTENZIONE: RhpcBLASctl non installato. Il BLAS potrebbe ignorare il tetto\n",
      "di thread impostato qui (legge le variabili d'ambiente solo all'avvio di R).\n",
      "Installa con install.packages(\"RhpcBLASctl\") e riavvia il kernel.\n", sep = "")
}
 
# --- funzioni dplyr fissate --------------------------------------------------
# Se in un'altra cella dello stesso kernel viene caricato plyr (es. da
# opt_center.R), le sue count/rename/mutate/... mascherano quelle di dplyr e
# rompono il codice in modo silenzioso o con errori poco leggibili
# ("object 'gene_name' not found", "Argument 'x' is not a vector: list").
# Qui si fissa esplicitamente la versione dplyr per tutto il file.
count     <- dplyr::count
rename    <- dplyr::rename
mutate    <- dplyr::mutate
summarise <- dplyr::summarise
summarize <- dplyr::summarize
arrange   <- dplyr::arrange
group_by  <- dplyr::group_by
filter    <- dplyr::filter
select    <- dplyr::select
desc      <- dplyr::desc
 
 
dir.create(PLOT_DIR, recursive = TRUE, showWarnings = FALSE)
 
.has <- function(p) !is.null(p) && length(p) == 1 && !is.na(p) && nzchar(p) && file.exists(p)

## 1. CARICAMENTO DATI  (da pipeline_v2.R, invariato)

In [ ]:
# ##############################################################################
# ##   1. CARICAMENTO DATI  (da pipeline_v2.R, invariato)                     ##
# ##############################################################################
 
sezione_inizio("1. Caricamento dati (sorgente:", DATA_SOURCE, ")")
 
if (DATA_SOURCE == "visium") {
 
  spe <- SpatialExperiment::read10xVisium(
    samples = INPUT_PATH, sample_id = SAMPLE_ID, type = VISIUM_TYPE,
    data = "filtered", images = "hires", load = TRUE)
 
  if (.has(POSITIONS_PATH)) {
    pos_t <- read_csv(POSITIONS_PATH, col_names = FALSE, show_col_types = FALSE)
    if (suppressWarnings(is.na(as.numeric(pos_t[[2]][1])))) pos_t <- pos_t[-1, ]
    names(pos_t)[1:2] <- c("Barcode", "in_tissue")
    flag <- as.numeric(pos_t$in_tissue)[match(colnames(spe), pos_t$Barcode)]
    n_off <- sum(!is.na(flag) & flag != 1)
    cat("Spot fuori dal tessuto rimossi:", n_off, "\n")
    if (n_off > 0) spe <- spe[, is.na(flag) | flag == 1]
  }
 
} else if (DATA_SOURCE == "rdata") {
 
  stopifnot(!is.null(RDATA_PATH))
  load(RDATA_PATH)
  stopifnot(exists("spots"), exists("counts"))
  bc <- intersect(c("Barcode", "barcode", "spot_id", "cell_id"), names(spots))[1]
  if (is.na(bc)) spots$Barcode <- colnames(counts) else names(spots)[names(spots) == bc] <- "Barcode"
  if ("cluster_annotation" %in% names(spots)) spots <- rename(spots, anatomy = cluster_annotation)
  if ("in_tissue" %in% names(spots)) spots <- filter(spots, in_tissue == 1)
  counts <- counts[, spots$Barcode, drop = FALSE]
  spe <- SpatialExperiment(assays = list(counts = as(counts, "dgCMatrix")),
                           colData = DataFrame(spots, row.names = spots$Barcode),
                           spatialCoordsNames = c("x", "y"))
} else stop("DATA_SOURCE deve essere 'visium' o 'rdata'.")
 
spe$sum_umi  <- colSums(counts(spe))
spe$sum_gene <- colSums(counts(spe) > 0)
cat("Geni:", nrow(spe), " Spot:", ncol(spe), "\n")

## 2. ANNOTAZIONE GENICA  (da pipeline_v2.R, invariato)

In [ ]:
# ##############################################################################
# ##   2. ANNOTAZIONE GENICA  (da pipeline_v2.R, invariato)                   ##
# ##############################################################################
 
sezione_inizio("2. Annotazione genica")
 
if (!is.null(GTF_PATH)) {
  gtf <- rtracklayer::import(GTF_PATH)
  gtf <- gtf[gtf$type == "gene"]
  gtf$gene_id <- gsub("\\..*", "", gtf$gene_id)
  gtf <- gtf[!duplicated(gtf$gene_id)]
  names(gtf) <- gtf$gene_id
 
  mg <- match(rownames(spe), gtf$gene_id)
  cat("Geni non annotati dal GTF:", sum(is.na(mg)), "\n")
  spe <- spe[!is.na(mg), ]; mg <- mg[!is.na(mg)]
 
  keep_cols <- intersect(c("source","type","gene_id","gene_name","gene_type","gene_biotype"),
                         colnames(mcols(gtf)))
  mcols(gtf) <- mcols(gtf)[, keep_cols]
  gtf$gene_name <- ifelse(is.na(gtf$gene_name) | gtf$gene_name == "", gtf$gene_id, gtf$gene_name)
  rowRanges(spe) <- gtf[mg]
  gene_table <- data.frame(gene_id = as.character(gtf$gene_id),
                           gene_name = as.character(gtf$gene_name), stringsAsFactors = FALSE)
} else {
  cat("GTF non fornito: gene_id = gene_name = rownames(spe)\n")
  rowData(spe)$gene_id   <- rownames(spe)
  rowData(spe)$gene_name <- rownames(spe)
  gene_table <- data.frame(gene_id = rownames(spe), gene_name = rownames(spe),
                           stringsAsFactors = FALSE)
}

## 3. METADATI DEGLI SPOT  (da pipeline_v2.R; niente inj_site)

In [ ]:
# ##############################################################################
# ##   3. METADATI DEGLI SPOT  (da pipeline_v2.R; niente inj_site)            ##
# ##############################################################################
 
sezione_inizio("3. Metadati degli spot")
 
coords <- as.data.frame(spatialCoords(spe))
spe$pixel_col <- coords[[1]]
spe$pixel_row <- coords[[2]]
 
if (!("anatomy" %in% names(colData(spe)) && any(!is.na(spe$anatomy)))) {
  if (.has(ANATOMY_PATH)) {
    a <- read_csv(ANATOMY_PATH, show_col_types = FALSE)
    spe$anatomy <- a[[2]][match(colnames(spe), a$Barcode)]
  } else spe$anatomy <- NA
}
cat("Spot senza anatomia:", sum(is.na(spe$anatomy)), "\n")
 
load_pseudotime <- function(path, expected_col, label) {
  if (!.has(path)) { cat("  ", label, ": file assente\n", sep = ""); return(rep(NA_real_, ncol(spe))) }
  d <- read_csv(path, show_col_types = FALSE); colnames(d)[1] <- "Barcode"
  vc <- if (expected_col %in% names(d)) {
    expected_col
  } else {
    setdiff(names(d)[sapply(d, is.numeric)], "Barcode")[1]
  }
  v <- d[[vc]][match(colnames(spe), d$Barcode)]
  cat("  ", label, ": ", sum(!is.na(v)), "/", ncol(spe), " spot matchati (colonna '", vc, "')\n", sep = "")
  v
}
spe$slingshot_pt <- load_pseudotime(SLINGSHOT_PATH, "pseudotime", "Slingshot")
spe$monocle_pt   <- load_pseudotime(MONOCLE_PATH,   "pseudotime", "Monocle")
spe$spaceflow_pt <- load_pseudotime(SPACEFLOW_PATH, "pseudotime", "SpaceFlow")
spe$isodepth     <- load_pseudotime(GASTON_PATH,    "isodepth",   "Gaston")

## 4. QUALITY CONTROL E NORMALIZZAZIONE  (da pipeline_v2.R, invariato)

In [ ]:
# ##############################################################################
# ##   4. QUALITY CONTROL E NORMALIZZAZIONE  (da pipeline_v2.R, invariato)    ##
# ##############################################################################
 
sezione_inizio("4. Quality control")
 
is_mito <- grepl("(^MT-)|(^mt-)", rowData(spe)$gene_name)
cat("Geni mitocondriali rimossi:", sum(is_mito), "\n")
spe <- addPerCellQC(spe, subsets = list(mito = is_mito))
spe <- spe[!is_mito, ]
 
if (any(!is.na(spe$anatomy))) {
  na_sp <- which(is.na(spe$anatomy))
  cat("Spot rimossi per anatomia mancante:", length(na_sp), "\n")
  if (length(na_sp) > 0) spe <- spe[, -na_sp, drop = FALSE]
}
 
no_c <- which(spe$sum == 0)
cat("Spot rimossi per 0 conteggi:", length(no_c), "\n")
if (length(no_c) > 0) spe <- spe[, -no_c, drop = FALSE]
 
keep_g <- Matrix::rowSums(counts(spe) > 0) >= MIN_SPOTS_PER_GENE
cat("Geni rimossi (espressi in <", MIN_SPOTS_PER_GENE, "spot):", sum(!keep_g), "\n")
spe <- spe[keep_g, , drop = FALSE]
 
spe <- scuttle::logNormCounts(spe)
logcounts_raw <- as.matrix(logcounts(spe))
 
spots <- as.data.frame(colData(spe))
spots$Barcode <- colnames(spe)
cat("Dimensioni finali:", nrow(spe), "geni x", ncol(spe), "spot\n")
 
# NOTA sul filtro geni: qui e' "espresso in >= 3 spot" (come v2). Il notebook
# di SpatialDE usa invece "conteggio totale >= 3". Filtri diversi, universi di
# geni leggermente diversi: se confronti i conteggi di significativi con il CSV
# di SpatialDE, la differenza parte gia' da qui.

## 6. COORDINATA RADIALE DAL BARCODE DI INIZIO

In [ ]:
# ##############################################################################
# ##   6. COORDINATA RADIALE DAL BARCODE DI INIZIO                            ##
# ##############################################################################
 
sezione_inizio("6. Coordinata radiale")
 
if (!(CENTER_BARCODE %in% colnames(spe)))
  stop("CENTER_BARCODE '", CENTER_BARCODE, "' non e' fra gli spot sopravvissuti al QC.")
 
center_idx  <- match(CENTER_BARCODE, colnames(spe))
CENTER_USED <- CENTER_BARCODE
cat("Centro:", CENTER_USED, "(indice attuale", center_idx, ")\n")
cat("  [v2 usava CENTER_INDEX = 1697, che qui e':", colnames(spe)[min(1697, ncol(spe))], "]\n")
 
r_center <- sqrt((spots$pixel_col - spots$pixel_col[center_idx])^2 +
                 (spots$pixel_row - spots$pixel_row[center_idx])^2)
 
cat("Raggio: min", round(min(r_center), 1), " max", round(max(r_center), 1),
    " mediana", round(median(r_center), 1), "px\n")
 
# I GLM radiali del blocco A (sezione 7) usano QUESTA coordinata.
dist_rgdist <- r_center

## 7. BLOCCO A - GLM_linear e GLM_gaussian

In [ ]:
# ##############################################################################
# ##   7. BLOCCO A - GLM_linear e GLM_gaussian                                ##
# ##                                                                          ##
# ##   GLM QL di edgeR con offset di libreria (sum_umi). Classifica: fdr,     ##
# ##   poi log p del gene (stesso ordine del p esatto, anche dove p = 0).     ##
# ##############################################################################
 
sezione_inizio("7. GLM radiale (lineare e gaussiano)")
 
d_scaled  <- dist_rgdist / max(dist_rgdist, na.rm = TRUE)
d2_scaled <- d_scaled^2
 
dge <- DGEList(counts = counts(spe))
dge$samples$norm.factors <- 1
dge$samples$lib.size     <- spe$sum_umi
 
fit_radial_glm <- function(design, coef_names, label, dge_obj = dge) {
  t_fit <- Sys.time()
  stampa("  ", label, " ... (estimateDisp + glmQLFit)")
  d_est <- estimateDisp(dge_obj, design)
  fit   <- glmQLFit(d_est, design)
  test  <- glmQLFTest(fit, coef = coef_names)
  tt    <- topTags(test, n = Inf, sort.by = "none")$table
 
  dev <- fit$deviance[match(rownames(tt), rownames(fit))]
 
  # log p per gli spareggi: log(PValue) dove PValue > 0; dove PValue e' 0 per
  # underflow, il log p della F di edgeR con i gradi di liberta' del gene
  i_tt   <- match(rownames(tt), rownames(test$table))
  log_pf <- pf(test$table$F, test$df.test, test$df.total, lower.tail = FALSE, log.p = TRUE)[i_tt]
  log_p  <- ifelse(tt$PValue > 0, log(tt$PValue), log_pf)
 
  out <- data.frame(gene_id = rownames(tt), p_value = tt$PValue, fdr = tt$FDR, log_p = log_p,
                    deviance = dev, n_par = ncol(design),
                    stringsAsFactors = FALSE)
  for (cn in coef_names) {
    col <- if (length(coef_names) == 1) tt$logFC else tt[[paste0("logFC.", cn)]]
    out[[paste0("coef_", gsub("[^A-Za-z0-9]", "", cn))]] <- if (is.null(col)) NA_real_ else col
  }
  stampa("  ", label, ": ", sum(out$fdr < ALPHA_FDR, na.rm = TRUE), " geni con FDR < ", ALPHA_FDR,
         " (", minuti_da(t_fit), " min)")
  out
}
 
# Classifica: fdr, poi la statistica del test (piu' piccola = piu' significativa).
# Con il log p i pareggi di fdr (blocchi BH, fdr = 0) seguono l'ordine del p esatto.
rank_by_fdr_then_stat <- function(fdr, stat) {
  r <- order(order(fdr, stat))
  r[is.na(fdr)] <- NA_integer_
  r
}
 
# Jaccard fra due insiemi di geni (usata in 9d.2 e 18; prima era nella 9b)
jac2 <- function(a, b) length(intersect(a, b)) / max(length(union(a, b)), 1)
 
design_lin <- model.matrix(~ d_scaled)
glm_lin    <- fit_radial_glm(design_lin, "d_scaled", "GLM_linear", dge_obj = dge)
 
design_gau <- model.matrix(~ d_scaled + d2_scaled)
glm_gau    <- fit_radial_glm(design_gau, c("d_scaled", "d2_scaled"), "GLM_gaussian", dge_obj = dge)
 
results <- list()
 
results$GLM_linear <- data.frame(
  gene_id = glm_lin$gene_id, cor = abs(glm_lin$coef_dscaled), fdr = glm_lin$fdr,
  rank = rank_by_fdr_then_stat(glm_lin$fdr, glm_lin$log_p),
  Method = "GLM_linear", stringsAsFactors = FALSE)
 
glm_gau_effect <- sqrt(glm_gau$coef_dscaled^2 + glm_gau$coef_d2scaled^2)
results$GLM_gaussian <- data.frame(
  gene_id = glm_gau$gene_id, cor = glm_gau_effect, fdr = glm_gau$fdr,
  rank = rank_by_fdr_then_stat(glm_gau$fdr, glm_gau$log_p),
  Method = "GLM_gaussian", stringsAsFactors = FALSE)
 
write_csv(glm_lin, file.path(OUTPUT_DIR, "glm_linear_full.csv"))
write_csv(glm_gau, file.path(OUTPUT_DIR, "glm_gaussian_full.csv"))

## 9c. RICERCA DEL CENTRO CON I GLM  (centro inferito)

In [ ]:
# ##############################################################################
# ##   9c. RICERCA DEL CENTRO CON I GLM  (centro inferito)                    ##
# ##                                                                          ##
# ##   Stessa idea di FindCenter_AI.ipynb: si gira la pipeline al contrario.  ##
# ##   Invece di fissare il centro e cercare i geni, si fissano i geni e si   ##
# ##   cerca, fra tutti gli spot, quello da cui la coordinata radiale spiega  ##
# ##   meglio l'espressione.                                                  ##
# ##                                                                          ##
# ##   COSA CAMBIA RISPETTO A FindCenter_AI                                   ##
# ##   Li' la cella della matrice [gene x candidato] era una Spearman fra     ##
# ##   log-ratio del gene e distanza dal candidato. Qui e' la statistica di   ##
# ##   un GLM binomiale negativo con lo stesso offset di library size della   ##
# ##   sezione 7: la profondita' e' quindi gia' dentro il modello, non serve  ##
# ##   ne' il log-ratio ne' il filtro sugli zeri.                             ##
# ##                                                                          ##
# ##   PERCHE' NON SI RIFA' IL FIT COMPLETO PER OGNI CANDIDATO                ##
# ##   estimateDisp + glmQLFit su ~2000 candidati x 2 forme sono centinaia di ##
# ##   ore. Qui si usa un test di punteggio ("score test"): il modello viene  ##
# ##   stimato UNA volta sola SENZA la distanza dentro (solo intercetta +     ##
# ##   offset), e per ogni candidato si misura quanto i residui di quel fit   ##
# ##   si allineano con la distanza da quel punto. E' la forma esatta della   ##
# ##   statistica che il GLM produrrebbe al valore nullo del coefficiente,    ##
# ##   e si calcola per tutti i geni e tutti i candidati insieme con qualche  ##
# ##   prodotto di matrici. Il fit vero e proprio si fa solo alla fine, sul   ##
# ##   centro vincitore (sezione 9d), e su un campione di candidati per       ##
# ##   verificare che lo scan sia un surrogato onesto (sezione 9c.5).         ##
# ##                                                                          ##
# ##   NOTA SULLA SOGLIA                                                      ##
# ##   FindCenter teneva le celle con |rho| > qnorm(0.975)/sqrt(n-1). Quella  ##
# ##   soglia e' esattamente |z| > 1.96: rho * sqrt(n-1) E' la z della        ##
# ##   Spearman. Qui la cella e' gia' una z, quindi la soglia e' la stessa    ##
# ##   identica cosa, scritta senza il passaggio di conversione.              ##
# ##                                                                          ##
# ##   NOTA SULLA SCALA                                                       ##
# ##   Il raggio viene normalizzato d/max(d) come nella sezione 7, e il max   ##
# ##   cambia da candidato a candidato. La statistica dello score test e'     ##
# ##   invariante al riscalamento della covariata, quindi il confronto fra    ##
# ##   candidati non risente di questo (confrontare coefficienti invece si'). ##
# ##############################################################################
 
# --- parametri di questa sezione ----------------------------------------------
# (se preferisci, spostali in cima nel blocco CONFIGURAZIONE: sono solo assegnazioni)
 
RUN_INFERRED_CENTER  <- TRUE
 
INFC_TOP_K           <- 100        # geni nel pannello (FindCenter_AI usa 100)
INFC_ALPHA_CELL      <- 0.05       # soglia di significativita' sulla singola cella
INFC_BLOCK           <- 250        # candidati processati per volta (memoria)
 
# Quale pannello di geni decide il centro usato a valle dalla sezione 9d:
#   "findcenter" : top-K per massimo sui candidati (fedele a FindCenter_AI)
#   "spatialde"  : top-K dal CSV di SpatialDE (kernel 2D su x/y, center-free)
INFC_PANEL_USED      <- "findcenter"
 
# Pannello nullo geometrico: stessi geni per livello di espressione ma senza
# segnale spaziale. Serve a vedere quanta parte della mappa e' pura geometria
# del tessuto e non biologia.
INFC_NULL_PANEL      <- TRUE
 
# Validazione dello scan: su quanti centri rifare il GLM VERO (fit_radial_glm).
# Costo = INFC_N_VALID x il tempo di un fit della sezione 7.
INFC_N_VALID         <- 25
INFC_VALID_SEED      <- 1
INFC_VALID_MODEL     <- "linear"   # "linear" | "both"  ("both" raddoppia il tempo)
 
# A valle: rifare i GLM dal centro inferito (sezione 9d).
RUN_INFERRED_CENTER_DOWNSTREAM <- TRUE
 
 
if (RUN_INFERRED_CENTER) {
 
sezione_inizio("9c. Ricerca del centro con i GLM (score test su tutti gli spot)")
 
 
# --- 9c.1  Fit nullo: una volta sola, senza la distanza -----------------------
#
# Modello: conteggi ~ intercetta, con offset di library size (sum_umi), lo
# stesso dge costruito nella sezione 7. Da qui escono due matrici [geni x spot]:
#   R = residuo pesato   -> quanto ogni spot si discosta da quello che il gene
#                           farebbe se non ci fosse nessuna struttura spaziale
#   W = peso             -> quanto quel residuo e' affidabile (tiene conto
#                           dell'overdispersione stimata per quel gene)
# Nessuna delle due dipende dal centro candidato: e' per questo che il fit si fa
# una volta e la scansione diventa algebra lineare.
 
cat("\n-- 9c.1 Fit nullo (intercetta + offset), una volta sola --\n")
 
design_null <- matrix(1, nrow = ncol(spe), ncol = 1,
                      dimnames = list(colnames(spe), "(Intercept)"))
 
t0 <- Sys.time()
stampa("   estimateDisp + glmFit sul modello nullo ...")
dge_null <- estimateDisp(dge, design_null)
fit_null <- glmFit(dge_null, design_null)
cat("   estimateDisp + glmFit: ", round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1),
    " minuti\n", sep = "")
 
phi_gene <- if (!is.null(dge_null$tagwise.dispersion)) dge_null$tagwise.dispersion else
            rep(dge_null$common.dispersion, nrow(dge_null))
 
mu_null <- fit_null$fitted.values          # [geni x spot], offset gia' incluso
Y_cnt   <- as.matrix(counts(spe))          # densificata una volta sola
 
denom <- 1 + phi_gene * mu_null
R_res <- (Y_cnt - mu_null) / denom         # residuo pesato
W_wts <- mu_null / denom                   # peso
 
rm(Y_cnt, mu_null, denom, fit_null); invisible(gc())
 
stopifnot(all(dim(R_res) == c(nrow(spe), ncol(spe))))
cat("   matrici R (residui) e W (pesi): ", nrow(R_res), " geni x ", ncol(R_res), " spot\n", sep = "")
cat("   dispersione: mediana ", signif(median(phi_gene), 4),
    "  (range ", signif(min(phi_gene), 3), " - ", signif(max(phi_gene), 3), ")\n", sep = "")
 
 
# --- 9c.2  Scansione di tutti gli spot candidati ------------------------------
#
# Per ogni candidato j si costruisce il raggio da j (normalizzato al suo massimo)
# e si calcolano due statistiche per ogni gene:
#   Z1 = forma lineare    -> z con segno, 1 grado di liberta'. E' l'analogo
#                            diretto della rho di FindCenter: |z| e' la stessa
#                            scala su cui li' era applicata la soglia.
#   X2 = forma gaussiana  -> chi-quadro a 2 gradi di liberta' (termine lineare +
#                            quadratico insieme), l'analogo di GLM_gaussian.
# Tutto per blocchi di candidati, per non tenere in RAM sette matrici
# [geni x tutti i candidati] contemporaneamente.
 
cat("\n-- 9c.2 Scansione degli spot candidati --\n")
 
coords_all <- cbind(spots$pixel_col, spots$pixel_row)
DM_all     <- as.matrix(dist(coords_all))        # [spot x candidato], simmetrica
 
n_gene_sc <- nrow(R_res)
n_spot_sc <- ncol(R_res)
n_cand_sc <- ncol(DM_all)
 
cat("   candidati: ", n_cand_sc, " (tutti gli spot sopravvissuti al QC)\n", sep = "")
cat("   blocchi da ", INFC_BLOCK, " -> ", ceiling(n_cand_sc / INFC_BLOCK), " passate\n", sep = "")
 
Z1_scan <- matrix(NA_real_, n_gene_sc, n_cand_sc,
                  dimnames = list(rownames(R_res), colnames(spe)))
X2_scan <- matrix(NA_real_, n_gene_sc, n_cand_sc,
                  dimnames = list(rownames(R_res), colnames(spe)))
 
# somma dei pesi per gene: non dipende dal candidato, si calcola una volta
S0_gene <- rowSums(W_wts)
 
t0 <- Sys.time()
for (start in seq(1, n_cand_sc, by = INFC_BLOCK)) {
 
  idx <- start:min(start + INFC_BLOCK - 1, n_cand_sc)
 
  X1 <- DM_all[, idx, drop = FALSE]
  X1 <- sweep(X1, 2, apply(X1, 2, max), "/")     # d / max(d), colonna per colonna
  X2p <- X1^2
  X3p <- X1^3
  X4p <- X1^4
 
  # allineamento residui-covariata (il "punteggio" del modello)
  U1 <- R_res %*% X1
  U2 <- R_res %*% X2p
 
  # momenti pesati della covariata, gene per gene
  S1 <- W_wts %*% X1
  S2 <- W_wts %*% X2p
  S3 <- W_wts %*% X3p
  S4 <- W_wts %*% X4p
 
  # --- forma lineare: 1 grado di liberta' --------------------------------------
  # La varianza del punteggio e' al netto dell'intercetta (che resta nel modello).
  V1 <- S2 - S1^2 / S0_gene
  V1[V1 <= 0] <- NA_real_
  Z1_scan[, idx] <- U1 / sqrt(V1)
 
  # --- forma gaussiana: 2 gradi di liberta' ------------------------------------
  # Stessa cosa con due covariate insieme (d e d^2): il punteggio e' un vettore
  # di due numeri e la varianza una matrice 2x2, sempre al netto dell'intercetta.
  a11 <- V1
  a12 <- S3 - S1 * S2 / S0_gene
  a22 <- S4 - S2^2 / S0_gene
  det2 <- a11 * a22 - a12^2
  det2[!is.finite(det2) | det2 <= 0] <- NA_real_
  X2_scan[, idx] <- (a22 * U1^2 - 2 * a12 * U1 * U2 + a11 * U2^2) / det2
 
  stampa("   candidati ", start, "-", max(idx), " fatti | ", stima_residua(max(idx), n_cand_sc, t0))
}
cat("   scansione completata in ",
    round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1), " minuti\n", sep = "")
 
Z1_scan[!is.finite(Z1_scan)] <- NA_real_
X2_scan[!is.finite(X2_scan)] <- NA_real_
 
rm(X1, X2p, X3p, X4p, U1, U2, S1, S2, S3, S4, V1, a11, a12, a22, det2)
invisible(gc())
 
# soglie sulla singola cella, coerenti coi gradi di liberta' delle due forme
SOGLIA_LIN <- qnorm(1 - INFC_ALPHA_CELL / 2)                   # 1.96, come FindCenter
SOGLIA_GAU <- sqrt(qchisq(1 - INFC_ALPHA_CELL, df = 2))        # 2.45
 
cat("   soglia per cella: forma lineare |z| > ", round(SOGLIA_LIN, 3),
    " | forma gaussiana sqrt(chi2_2) > ", round(SOGLIA_GAU, 3), "\n", sep = "")
 
 
# --- 9c.3  Pannelli di geni ---------------------------------------------------
#
# Il punteggio di uno spot e' la mediana della statistica sui geni del pannello.
# Quali geni ci finiscono cambia la risposta, quindi si calcolano due pannelli
# indipendenti e si confrontano le mappe.
#
#   A "findcenter" : i K geni col massimo |z| su TUTTI i candidati. Fedele a
#                    FindCenter_AI. Si sceglie i geni guardando lo stesso scan
#                    che poi produce la mappa: e' auto-selezione, e premia i
#                    geni con gradiente monotono su tutta la fetta.
#   B "spatialde"  : i K geni piu' significativi del CSV di SpatialDE, che usa
#                    un kernel 2D su x/y e non sa nulla di centri. E' un
#                    pannello scelto fuori da questo scan: se le due mappe
#                    indicano lo stesso spot, il risultato non dipende dalla
#                    selezione.
#   N "nullo"      : geni senza segnale spaziale, appaiati per livello di
#                    espressione ai geni del pannello A. La sua mappa e' la
#                    componente puramente geometrica (quanto uno spot "vince"
#                    solo per dove si trova nel tessuto).
 
cat("\n-- 9c.3 Pannelli di geni --\n")
 
max_z_gene <- apply(abs(Z1_scan), 1, function(x) if (all(is.na(x))) NA_real_ else max(x, na.rm = TRUE))
 
# pannello A: auto-selezione dallo scan
pannello_A <- order(max_z_gene, decreasing = TRUE, na.last = NA)[seq_len(INFC_TOP_K)]
cat("   pannello A (findcenter): ", length(pannello_A), " geni, |z| max da ",
    round(min(max_z_gene[pannello_A]), 2), " a ", round(max(max_z_gene[pannello_A]), 2), "\n", sep = "")
 
# pannello B: center-free, dal CSV di SpatialDE
# (lettura autonoma: la sezione 10b legge lo stesso file ma viene dopo, e questo
#  blocco deve poter girare da solo)
pannello_B <- integer(0)
if (!is.null(SPATIALDE_PATH) && .has(SPATIALDE_PATH)) {
 
  sde_panel <- read_csv(SPATIALDE_PATH, show_col_types = FALSE)
  gcol_p <- intersect(c("gene", "g", "gene_id", "...1"), names(sde_panel))[1]
  qcol_p <- intersect(c("qval", "q_value", "padj", "FDR"), names(sde_panel))[1]
 
  if (!is.na(gcol_p) && !is.na(qcol_p)) {
    sde_panel <- sde_panel %>% rename(gene_id = !!gcol_p, qval = !!qcol_p)
    sde_panel$gene_id <- sub("\\..*$", "", as.character(sde_panel$gene_id))
 
    ov_id_p  <- length(intersect(sde_panel$gene_id, rownames(spe)))
    ov_sym_p <- length(intersect(tolower(sde_panel$gene_id),
                                 tolower(as.character(rowData(spe)$gene_name))))
    if (ov_sym_p > ov_id_p) {
      sm_p <- gene_table[!duplicated(tolower(gene_table$gene_name)), ]
      sde_panel$gene_id <- sm_p$gene_id[match(tolower(sde_panel$gene_id), tolower(sm_p$gene_name))]
      sde_panel <- sde_panel[!is.na(sde_panel$gene_id), ]
    }
 
    sde_panel  <- sde_panel[order(sde_panel$qval), ]
    geni_B     <- intersect(sde_panel$gene_id, rownames(Z1_scan))[seq_len(INFC_TOP_K)]
    geni_B     <- geni_B[!is.na(geni_B)]
    pannello_B <- match(geni_B, rownames(Z1_scan))
 
    cat("   pannello B (spatialde) : ", length(pannello_B), " geni\n", sep = "")
    cat("   sovrapposizione A/B    : ",
        length(intersect(pannello_A, pannello_B)), " geni su ", INFC_TOP_K, "\n", sep = "")
  }
}
if (length(pannello_B) == 0)
  cat("   pannello B NON disponibile (CSV SpatialDE assente o illeggibile)\n")
 
# pannello nullo: appaiato per espressione media, preso dalla meta' bassa di max|z|
pannello_N <- integer(0)
if (INFC_NULL_PANEL) {
  espr_media <- rowMeans(logcounts_raw)
  candidati_nulli <- which(max_z_gene <= median(max_z_gene, na.rm = TRUE))
  if (length(candidati_nulli) >= INFC_TOP_K) {
    for (g in pannello_A) {
      libero <- setdiff(candidati_nulli, pannello_N)
      pannello_N <- c(pannello_N, libero[which.min(abs(espr_media[libero] - espr_media[g]))])
    }
    cat("   pannello N (nullo)     : ", length(pannello_N),
        " geni, espressione media appaiata al pannello A\n", sep = "")
  }
}
 
 
# --- 9c.4  Mappe del centro ---------------------------------------------------
#
# Identica alla riduzione di FindCenter_AI: si azzerano (NA) le celle non
# significative e si prende la mediana per spot sui geni del pannello.
# Il massimo della mappa e' il centro stimato.
 
cat("\n-- 9c.4 Mappe --\n")
 
mappa_centro <- function(STAT, soglia, geni) {
  if (length(geni) == 0) return(rep(NA_real_, ncol(STAT)))
  M <- STAT[geni, , drop = FALSE]
  M[M <= soglia] <- NA_real_
  apply(M, 2, median, na.rm = TRUE)
}
 
ABS_Z  <- abs(Z1_scan)
SQRT_X <- sqrt(X2_scan)
 
mappe <- list(
  A_lin = mappa_centro(ABS_Z,  SOGLIA_LIN, pannello_A),
  A_gau = mappa_centro(SQRT_X, SOGLIA_GAU, pannello_A),
  B_lin = mappa_centro(ABS_Z,  SOGLIA_LIN, pannello_B),
  B_gau = mappa_centro(SQRT_X, SOGLIA_GAU, pannello_B),
  N_lin = mappa_centro(ABS_Z,  SOGLIA_LIN, pannello_N)
)
mappe <- mappe[!sapply(mappe, function(m) all(is.na(m)))]
 
centroide <- c(mean(spots$pixel_col), mean(spots$pixel_row))
d_centroide <- sqrt((spots$pixel_col - centroide[1])^2 + (spots$pixel_row - centroide[2])^2)
 
riassunto_mappe <- bind_rows(lapply(names(mappe), function(nm) {
  m  <- mappe[[nm]]
  j  <- which.max(m)
  data.frame(
    mappa            = nm,
    barcode          = colnames(spe)[j],
    indice           = j,
    punteggio        = round(m[j], 3),
    rango_del_barcode_iniziale = rank(-m, na.last = "keep")[center_idx],
    punteggio_barcode_iniziale = round(m[center_idx], 3),
    dist_da_barcode_px = round(sqrt((spots$pixel_col[j] - spots$pixel_col[center_idx])^2 +
                                    (spots$pixel_row[j] - spots$pixel_row[center_idx])^2), 1),
    dist_da_centroide_px = round(d_centroide[j], 1),
    percentile_dist_centroide = round(mean(d_centroide <= d_centroide[j]), 3),
    stringsAsFactors = FALSE)
}))
 
cat("\n>>> CENTRI STIMATI <<<\n")
print(riassunto_mappe, row.names = FALSE)
 
cat("\nLettura della colonna percentile_dist_centroide: 1.00 significa che il\n")
cat("centro stimato e' lo spot piu' lontano dal baricentro del tessuto, cioe' sul\n")
cat("bordo. Un centro al bordo e' il modo tipico in cui questa ricerca fallisce:\n")
cat("da un angolo il raggio cresce monotono su tutta la fetta, quindi qualunque\n")
cat("gene con un gradiente ci si allinea benissimo anche senza nessun centro vero.\n")
 
# accordo fra le mappe
if (length(mappe) > 1) {
  cat("\nCorrelazione di Spearman fra le mappe (accordo sulla forma, non solo sul massimo):\n")
  cor_mappe <- round(cor(do.call(cbind, mappe), method = "spearman", use = "pairwise.complete.obs"), 3)
  print(cor_mappe)
}
if ("N_lin" %in% names(mappe)) {
  cat("\nLa riga/colonna N_lin e' la mappa del pannello NULLO. Se correla molto con\n")
  cat("A_lin, gran parte del rilievo della mappa e' geometria del tessuto e non\n")
  cat("segnale: in quel caso il massimo va preso con molta cautela.\n")
}
 
# mappa in pixel, una figura per mappa (stesso stile di FindCenter_AI)
dir.create(PLOT_DIR, showWarnings = FALSE, recursive = TRUE)
for (nm in names(mappe)) {
  md <- data.frame(x = spots$pixel_col, y = spots$pixel_row, punteggio = mappe[[nm]])
  jm <- which.max(mappe[[nm]])
  p_map <- ggplot(md, aes(x = x, y = y, fill = punteggio)) +
    geom_point(shape = 21, size = 2.6, stroke = 0.15) +
    geom_point(data = md[jm, ], shape = 21, size = 5, stroke = 1.6, fill = NA, colour = "black") +
    geom_point(data = md[center_idx, ], shape = 4, size = 4, stroke = 1.4, colour = "white") +
    scale_fill_gradientn(colors = c("blue", "cyan", "lightgreen", "gold", "darkorange", "red"),
                         na.value = "grey85") +
    scale_y_reverse() + theme_minimal() +
    labs(title = paste0("Origine della variabilita' - mappa ", nm),
         subtitle = "cerchio nero = massimo della mappa   |   x bianca = CENTER_BARCODE",
         x = "Pixel colonna", y = "Pixel riga", fill = "mediana stat.")
  ggsave(file.path(PLOT_DIR, paste0("09c_mappa_centro_", nm, ".png")),
         p_map, width = 7, height = 6, dpi = 150)
  print(p_map)
}
 
write_csv(riassunto_mappe, file.path(OUTPUT_DIR, "inferred_center_riassunto_mappe.csv"))
write_csv(data.frame(barcode = colnames(spe),
                     pixel_col = spots$pixel_col, pixel_row = spots$pixel_row,
                     as.data.frame(do.call(cbind, mappe))),
          file.path(OUTPUT_DIR, "inferred_center_mappe_per_spot.csv"))
 
 
# --- 9c.5  Validazione: lo scan predice davvero il GLM vero? ------------------
#
# Lo score test e' una statistica al valore nullo del coefficiente, il benchmark
# usa il QL F-test di edgeR con shrinkage: non sono la stessa quantita'. Qui si
# prendono alcuni candidati (il vincitore, il barcode di partenza, e il resto a
# caso), si rifa' su ciascuno il fit COMPLETO della sezione 7, e si guarda se
# l'ordine dei candidati secondo lo scan e' lo stesso che darebbe il fit vero.
# Se la correlazione e' alta, lo scan e' un surrogato utilizzabile. Se e' bassa,
# la mappa non va usata per scegliere il centro.
 
cat("\n-- 9c.5 Validazione dello scan contro il GLM completo --\n")
 
mappa_scelta <- if (INFC_PANEL_USED == "spatialde" && "B_lin" %in% names(mappe))
                  mappe$B_lin else mappe$A_lin
 
if (is.null(mappa_scelta) || all(is.na(mappa_scelta)))
  stop("Nessuna mappa utilizzabile: tutte le celle del pannello sono sotto soglia. ",
       "Abbassa INFC_ALPHA_CELL o allarga INFC_TOP_K prima di proseguire.")
 
set.seed(INFC_VALID_SEED)
cand_valid <- unique(c(
  which.max(mappa_scelta),
  center_idx,
  unlist(lapply(names(mappe), function(nm) which.max(mappe[[nm]]))),
  sample(seq_len(n_cand_sc), max(0, INFC_N_VALID - 2 - length(mappe)))
))
cand_valid <- cand_valid[seq_len(min(length(cand_valid), INFC_N_VALID))]
 
cat("   ", length(cand_valid), " centri da rifittare (modello: ", INFC_VALID_MODEL, ")\n", sep = "")
cat("   tempo atteso: ", length(cand_valid), " x il tempo di un fit della sezione 7\n", sep = "")
 
# fit_radial_glm costruisce il design con model.matrix(~ d_scaled), che cerca
# d_scaled nell'ambiente globale: lo si imposta prima di ogni chiamata e lo si
# rimette a posto alla fine del blocco.
d_scaled_backup  <- d_scaled
d2_scaled_backup <- d2_scaled
 
valid_rows <- list()
t0_valid <- Sys.time()
for (k_valid in seq_along(cand_valid)) {
  j <- cand_valid[k_valid]
  stampa("   validazione: centro ", k_valid, "/", length(cand_valid), " (", colnames(spe)[j], ")")
 
  r_j <- sqrt((spots$pixel_col - spots$pixel_col[j])^2 +
              (spots$pixel_row - spots$pixel_row[j])^2)
 
  d_scaled  <- r_j / max(r_j)
  d2_scaled <- d_scaled^2
 
  f_lin <- fit_radial_glm(model.matrix(~ d_scaled), "d_scaled",
                          paste0("valid_lin_", colnames(spe)[j]), dge_obj = dge)
  n_sig_lin_j <- sum(f_lin$fdr < ALPHA_FDR, na.rm = TRUE)
 
  n_sig_gau_j <- NA_integer_
  if (INFC_VALID_MODEL == "both") {
    f_gau <- fit_radial_glm(model.matrix(~ d_scaled + d2_scaled), c("d_scaled", "d2_scaled"),
                            paste0("valid_gau_", colnames(spe)[j]), dge_obj = dge)
    n_sig_gau_j <- sum(f_gau$fdr < ALPHA_FDR, na.rm = TRUE)
  }
 
  valid_rows[[length(valid_rows) + 1]] <- data.frame(
    indice = j, barcode = colnames(spe)[j],
    punteggio_scan   = mappa_scelta[j],
    scan_media_absz  = mean(ABS_Z[pannello_A, j], na.rm = TRUE),
    n_sig_glm_lin    = n_sig_lin_j,
    n_sig_glm_gau    = n_sig_gau_j,
    e_il_vincitore   = (j == which.max(mappa_scelta)),
    e_il_barcode     = (j == center_idx),
    stringsAsFactors = FALSE)
  stampa("   validazione: ", stima_residua(k_valid, length(cand_valid), t0_valid))
}
 
d_scaled  <- d_scaled_backup
d2_scaled <- d2_scaled_backup
 
valid_tab <- bind_rows(valid_rows)
valid_tab <- valid_tab[order(-valid_tab$punteggio_scan), ]
 
cat("\n>>> SCAN vs GLM COMPLETO <<<\n")
print(valid_tab, row.names = FALSE)
 
rho_valid <- suppressWarnings(cor(valid_tab$punteggio_scan, valid_tab$n_sig_glm_lin,
                                  method = "spearman", use = "complete.obs"))
cat("\nSpearman fra punteggio dello scan e geni significativi del GLM vero: ",
    round(rho_valid, 3), "\n", sep = "")
if (!is.na(rho_valid) && rho_valid >= 0.8) {
  cat("-> lo scan ordina i centri come il GLM completo: surrogato accettabile.\n")
} else {
  cat("-> ATTENZIONE: lo scan NON riproduce l'ordine del GLM completo. Il centro\n")
  cat("   trovato in 9c.4 non e' difendibile; prima di usarlo a valle va capito\n")
  cat("   perche' (candidati troppo simili fra loro, dispersione mal stimata sotto\n")
  cat("   il modello nullo, o segnale troppo debole per ordinare qualcosa).\n")
}
 
write_csv(valid_tab, file.path(OUTPUT_DIR, "inferred_center_validazione_scan.csv"))
 
 
# --- 9c.6  Il centro inferito ---------------------------------------------------
 
CENTER_INFERRED_IDX <- which.max(mappa_scelta)
CENTER_INFERRED     <- colnames(spe)[CENTER_INFERRED_IDX]
 
cat("\n>>> CENTRO INFERITO <<<\n")
cat("Pannello usato : ", INFC_PANEL_USED, "\n", sep = "")
cat("Barcode        : ", CENTER_INFERRED, " (indice ", CENTER_INFERRED_IDX, ")\n", sep = "")
cat("Barcode iniziale: ", CENTER_BARCODE, " (indice ", center_idx, ")\n", sep = "")
cat("Distanza fra i due centri: ",
    round(sqrt((spots$pixel_col[CENTER_INFERRED_IDX] - spots$pixel_col[center_idx])^2 +
               (spots$pixel_row[CENTER_INFERRED_IDX] - spots$pixel_row[center_idx])^2), 1), " px\n", sep = "")
 
# le due matrici grandi non servono piu' a valle
rm(R_res, W_wts, ABS_Z, SQRT_X, DM_all); invisible(gc())
 
} else {
  sezione_inizio("9c. Ricerca del centro SALTATA (RUN_INFERRED_CENTER = FALSE)")
  CENTER_INFERRED <- NULL
}

## 9d. GLM RIFATTI DAL CENTRO INFERITO

In [ ]:
# ##############################################################################
# ##   9d. GLM RIFATTI DAL CENTRO INFERITO                                    ##
# ##                                                                          ##
# ##   Il centro non sostituisce quello di partenza: diventa un secondo        ##
# ##   livello del fattore "centro". I metodi della sezione 7 restano          ##
# ##   esattamente dove sono, con gli stessi nomi; qui se ne aggiungono due    ##
# ##   col suffisso _inferred:                                                ##
# ##     GLM_linear_inferred, GLM_gaussian_inferred                           ##
# ##                                                                          ##
# ##   AVVERTENZA DA TENERE A MENTE LEGGENDO I RISULTATI                       ##
# ##   Il centro inferito e' stato scelto PER massimizzare il segnale dei GLM. ##
# ##   Che GLM_*_inferred trovi piu' geni significativi di GLM_* non dimostra  ##
# ##   niente: e' vero per costruzione, anche su dati senza struttura. L'unico ##
# ##   confronto che dice qualcosa e' su un criterio esterno alla selezione:   ##
# ##   Golden Standard (sezione 12) e pathway (sezioni 14-15).                 ##
# ##############################################################################
 
if (RUN_INFERRED_CENTER && RUN_INFERRED_CENTER_DOWNSTREAM && !is.null(CENTER_INFERRED)) {
 
sezione_inizio("9d. GLM dal centro inferito (", CENTER_INFERRED, ")", sep = "")
 
r_inferred <- sqrt((spots$pixel_col - spots$pixel_col[CENTER_INFERRED_IDX])^2 +
                   (spots$pixel_row - spots$pixel_row[CENTER_INFERRED_IDX])^2)
 
cat("Raggio dal centro inferito: min ", round(min(r_inferred), 1),
    "  max ", round(max(r_inferred), 1),
    "  mediana ", round(median(r_inferred), 1), " px\n", sep = "")
cat("(dal barcode iniziale era: min ", round(min(r_center), 1),
    "  max ", round(max(r_center), 1),
    "  mediana ", round(median(r_center), 1), " px)\n", sep = "")
cat("Correlazione di Spearman fra le due coordinate radiali: ",
    round(cor(r_inferred, r_center, method = "spearman"), 3), "\n", sep = "")
 
 
# --- 9d.1 GLM lineare e gaussiano dal centro inferito -------------------------
# Stesso identico fit_radial_glm della sezione 7, stesso dge, stesso offset.
# Cambia solo il contenuto di d_scaled.
 
cat("\n-- 9d.1 GLM --\n")
 
d_scaled_backup_9d  <- d_scaled
d2_scaled_backup_9d <- d2_scaled
 
d_scaled  <- r_inferred / max(r_inferred, na.rm = TRUE)
d2_scaled <- d_scaled^2
 
design_lin_inferred <- model.matrix(~ d_scaled)
glm_lin_inferred    <- fit_radial_glm(design_lin_inferred, "d_scaled", "GLM_linear_inferred", dge_obj = dge)
 
design_gau_inferred <- model.matrix(~ d_scaled + d2_scaled)
glm_gau_inferred    <- fit_radial_glm(design_gau_inferred, c("d_scaled", "d2_scaled"),
                                      "GLM_gaussian_inferred", dge_obj = dge)
 
results$GLM_linear_inferred <- data.frame(
  gene_id = glm_lin_inferred$gene_id, cor = abs(glm_lin_inferred$coef_dscaled),
  fdr = glm_lin_inferred$fdr,
  rank = rank_by_fdr_then_stat(glm_lin_inferred$fdr, glm_lin_inferred$log_p),
  Method = "GLM_linear_inferred", stringsAsFactors = FALSE)
 
glm_gau_inferred_effect <- sqrt(glm_gau_inferred$coef_dscaled^2 + glm_gau_inferred$coef_d2scaled^2)
results$GLM_gaussian_inferred <- data.frame(
  gene_id = glm_gau_inferred$gene_id, cor = glm_gau_inferred_effect, fdr = glm_gau_inferred$fdr,
  rank = rank_by_fdr_then_stat(glm_gau_inferred$fdr, glm_gau_inferred$log_p),
  Method = "GLM_gaussian_inferred", stringsAsFactors = FALSE)
 
write_csv(glm_lin_inferred, file.path(OUTPUT_DIR, "glm_linear_inferred_full.csv"))
write_csv(glm_gau_inferred, file.path(OUTPUT_DIR, "glm_gaussian_inferred_full.csv"))
 
# ripristino: d_scaled torna al barcode iniziale, cosi' un ri-lancio della
# sezione 7 o della 10c non usa per sbaglio il raggio inferito
d_scaled  <- d_scaled_backup_9d
d2_scaled <- d2_scaled_backup_9d
 
 
# --- 9d.2 Confronto barcode vs centro inferito --------------------------------
 
cat("\n-- 9d.2 Barcode iniziale vs centro inferito --\n")
 
confronto_centri <- data.frame(
  Metodo = c("GLM_linear", "GLM_gaussian"),
  n_sig_barcode = c(sum(glm_lin$fdr      < ALPHA_FDR, na.rm = TRUE),
                    sum(glm_gau$fdr      < ALPHA_FDR, na.rm = TRUE)),
  n_sig_inferred = c(sum(glm_lin_inferred$fdr < ALPHA_FDR, na.rm = TRUE),
                     sum(glm_gau_inferred$fdr < ALPHA_FDR, na.rm = TRUE)),
  stringsAsFactors = FALSE)
confronto_centri$variazione_pct <- round(
  100 * (confronto_centri$n_sig_inferred - confronto_centri$n_sig_barcode) /
    pmax(confronto_centri$n_sig_barcode, 1), 1)
 
# Jaccard fra gli insiemi significativi: dice se i due centri trovano gli stessi
# geni o geni diversi, che e' piu' informativo del solo conteggio
jac_centri <- c(
  jac2(glm_lin$gene_id[which(glm_lin$fdr < ALPHA_FDR)],
       glm_lin_inferred$gene_id[which(glm_lin_inferred$fdr < ALPHA_FDR)]),
  jac2(glm_gau$gene_id[which(glm_gau$fdr < ALPHA_FDR)],
       glm_gau_inferred$gene_id[which(glm_gau_inferred$fdr < ALPHA_FDR)]))
confronto_centri$jaccard <- round(jac_centri, 3)
 
cat("\n>>> CENTRO: BARCODE vs INFERITO <<<\n")
print(confronto_centri, row.names = FALSE)
 
cat("\nPROMEMORIA: la colonna n_sig_inferred dei due GLM e' gonfiata per costruzione.\n")
cat("Il centro e' stato scelto massimizzando esattamente quella quantita', quindi\n")
cat("un aumento non e' una prova che il centro sia migliore. Guarda il Golden\n")
cat("Standard (sezione 12) e i pathway (14-15), e la colonna jaccard: se e' alta,\n")
cat("i due centri stanno raccontando la stessa storia e la scelta conta poco.\n")
 
write_csv(confronto_centri, file.path(OUTPUT_DIR, "inferred_center_confronto_barcode.csv"))
 
} else if (RUN_INFERRED_CENTER) {
  sezione_inizio("9d. SALTATA (RUN_INFERRED_CENTER_DOWNSTREAM = FALSE)")
}

## 10. BLOCCO D - BENCHMARK A PARTE: GLM CON PSEUDOTIME

In [ ]:
# ##############################################################################
# ##   10. BLOCCO D - BENCHMARK A PARTE: GLM CON PSEUDOTIME                   ##
# ##                                                                          ##
# ##   Stesso identico fit_radial_glm del blocco A. Cambia solo la covariata: ##
# ##   il pseudotime riscalato a [0,1] prende il posto della distanza.        ##
# ##   Se il pseudotime ha NA, il DGEList viene ricostruito sul sottoinsieme  ##
# ##   di spot usati, altrimenti l'offset di libreria e' disallineato.        ##
# ##############################################################################
 
if (RUN_GLM_PSEUDOTIME) {
 
sezione_inizio("10. Benchmark separato: GLM su pseudotime")
 
PT_LIST <- list(
  Gaston    = spe$isodepth,
  SpaceFlow = spe$spaceflow_pt,
  Slingshot = spe$slingshot_pt,
  Monocle   = spe$monocle_pt
)
 
glm_pt_results <- list()
glm_pt_tab     <- list()
t0_pt <- Sys.time()
 
for (pt_name in names(PT_LIST)) {
 
  stampa("  pseudotime ", match(pt_name, names(PT_LIST)), "/", length(PT_LIST), ": ", pt_name,
         " | trascorsi ", minuti_da(t0_pt), " min")
  v  <- PT_LIST[[pt_name]]
  ok <- is.finite(v)   # esclude NA, NaN, Inf, -Inf: is.na() da solo lasciava
                        # passare gli Inf e faceva collassare min()/max() sotto,
                        # con model.matrix() che poi droppava righe NaN in
                        # silenzio (mismatch design vs dge in glmFit).
 
  n_non_finiti <- sum(!is.na(v) & !is.finite(v))
  if (n_non_finiti > 0)
    cat("  ", pt_name, ": ", n_non_finiti, " valori non finiti (Inf/-Inf) rimossi\n", sep = "")
 
  if (sum(ok) < GLM_PT_MIN_SPOTS) {
    cat("  ", pt_name, ": solo ", sum(ok), " spot finiti, saltato\n", sep = "")
    next
  }
  cat("  ", pt_name, " (", sum(ok), " spot su ", ncol(spe), ")\n", sep = "")
 
  x <- v[ok]
  if (max(x) == min(x)) {
    cat("  ", pt_name, ": pseudotime senza varianza (max == min), saltato\n", sep = "")
    next
  }
  d_scaled  <- (x - min(x)) / (max(x) - min(x))
  d2_scaled <- d_scaled^2
 
  dge_pt <- DGEList(counts = counts(spe)[, ok, drop = FALSE])
  dge_pt$samples$norm.factors <- 1
  dge_pt$samples$lib.size     <- spe$sum_umi[ok]
 
  des_l <- model.matrix(~ d_scaled)
  if (nrow(des_l) != ncol(dge_pt))
    stop(pt_name, ": nrow(design)=", nrow(des_l), " vs ncol(dge_pt)=", ncol(dge_pt),
        " - model.matrix() ha droppato righe (verosimilmente NA/NaN residui in d_scaled).")
  g_lin <- fit_radial_glm(des_l, "d_scaled",
                          paste0("GLM_linear_", pt_name), dge_obj = dge_pt)
 
  des_g <- model.matrix(~ d_scaled + d2_scaled)
  g_gau <- fit_radial_glm(des_g, c("d_scaled", "d2_scaled"),
                          paste0("GLM_gaussian_", pt_name), dge_obj = dge_pt)
 
  glm_pt_results[[paste0("GLM_linear_", pt_name)]] <- data.frame(
    gene_id = g_lin$gene_id, cor = abs(g_lin$coef_dscaled), fdr = g_lin$fdr,
    rank = rank_by_fdr_then_stat(g_lin$fdr, g_lin$log_p),
    Method = paste0("GLM_linear_", pt_name), stringsAsFactors = FALSE)
 
  eff_g <- sqrt(g_gau$coef_dscaled^2 + g_gau$coef_d2scaled^2)
  glm_pt_results[[paste0("GLM_gaussian_", pt_name)]] <- data.frame(
    gene_id = g_gau$gene_id, cor = eff_g, fdr = g_gau$fdr,
    rank = rank_by_fdr_then_stat(g_gau$fdr, g_gau$log_p),
    Method = paste0("GLM_gaussian_", pt_name), stringsAsFactors = FALSE)
 
  glm_pt_tab[[pt_name]] <- data.frame(
    Covariata   = pt_name,
    n_spot      = sum(ok),
    cor_profondita = round(cor(x, spe$sum_umi[ok], method = "spearman"), 3),
    n_sig_lin   = sum(g_lin$fdr < ALPHA_FDR, na.rm = TRUE),
    n_sig_gau   = sum(g_gau$fdr < ALPHA_FDR, na.rm = TRUE),
    stringsAsFactors = FALSE)
}
 
# Il ciclo qui sopra sovrascrive d_scaled/d2_scaled (model.matrix li cerca
# nell'ambiente globale): li riporto ai valori radiali del blocco A, cosi' un
# eventuale ri-lancio dello step 7 non usa per sbaglio l'ultimo pseudotime.
d_scaled  <- dist_rgdist / max(dist_rgdist, na.rm = TRUE)
d2_scaled <- d_scaled^2
 
pt_summary <- bind_rows(
  data.frame(Covariata = "distanza dal barcode (riferimento)",
             n_spot = ncol(spe),
             cor_profondita = round(cor(dist_rgdist, spe$sum_umi, method = "spearman"), 3),
             n_sig_lin = sum(glm_lin$fdr < ALPHA_FDR, na.rm = TRUE),
             n_sig_gau = sum(glm_gau$fdr < ALPHA_FDR, na.rm = TRUE),
             stringsAsFactors = FALSE),
  bind_rows(glm_pt_tab))
 
cat("\n>>> GLM: DISTANZA vs PSEUDOTIME <<<\n")
print(pt_summary, row.names = FALSE)
cat("\nNota: cor_profondita e' la Spearman fra covariata e sum_umi. Nel GLM la\n")
cat("profondita' entra come offset di libreria, non come termine del design:\n")
cat("una covariata molto correlata con la profondita' resta quindi il caso\n")
cat("piu' delicato da interpretare, anche qui.\n")
 
write_csv(pt_summary, file.path(OUTPUT_DIR, "glm_pseudotime_summary.csv"))
write_csv(bind_rows(glm_pt_results), file.path(OUTPUT_DIR, "glm_pseudotime_long.csv"))
 
# Uniti alla lista globale: da qui in poi i metodi su pseudotime entrano in
# Golden Standard, boxplot, ROC, ORA, GSEA insieme ai 4 principali. Restano
# distinguibili tramite la colonna Group definita nella sezione 11.
results <- c(results, glm_pt_results)
 
} else {
  glm_pt_results <- list()
  sezione_inizio("10. Benchmark GLM su pseudotime SALTATO (RUN_GLM_PSEUDOTIME = FALSE)")
}

## 10b. SpatialDE ORIGINALE  (step 10 di v2, invariato)

In [ ]:
# ##############################################################################
# ##   10b. SpatialDE ORIGINALE  (step 10 di v2, invariato)                   ##
# ##                                                                          ##
# ##   Risultati del notebook Python: kernel 2D sulle coordinate x/y, non     ##
# ##   sul raggio. E' il riferimento esterno vero e proprio - l'unico metodo  ##
# ##   qui che non e' radiale. Non riceve la correzione per profondita'       ##
# ##   perche' la applica gia' internamente (Svensson et al. 2018).          ##
# ##############################################################################
 
HAS_SDE_ORIG <- !is.null(SPATIALDE_PATH) && .has(SPATIALDE_PATH)
 
if (HAS_SDE_ORIG) {
  sezione_inizio("10b. SpatialDE originale (da CSV)")
 
  sde_orig <- read_csv(SPATIALDE_PATH, show_col_types = FALSE)
  gcol <- intersect(c("gene", "g", "gene_id", "...1"), names(sde_orig))[1]
  qcol <- intersect(c("qval", "q_value", "padj", "FDR"), names(sde_orig))[1]
  stopifnot(!is.na(gcol), !is.na(qcol))
 
  sde_orig <- sde_orig %>% rename(gene_id = !!gcol, qval = !!qcol)
  sde_orig$gene_id <- sub("\\..*$", "", as.character(sde_orig$gene_id))
 
  ov_id  <- length(intersect(sde_orig$gene_id, rownames(spe)))
  ov_sym <- length(intersect(tolower(sde_orig$gene_id), tolower(as.character(rowData(spe)$gene_name))))
  if (ov_sym > ov_id) {
    cat("Il file SpatialDE contiene simboli: converto in Ensembl ID\n")
    sm <- gene_table[!duplicated(tolower(gene_table$gene_name)), ]
    sde_orig$gene_id <- sm$gene_id[match(tolower(sde_orig$gene_id), tolower(sm$gene_name))]
    sde_orig <- sde_orig[!is.na(sde_orig$gene_id), ]
  }
  if (!"LLR" %in% names(sde_orig))
    sde_orig$LLR <- -log10(pmax(sde_orig$qval, .Machine$double.xmin))
 
  # I rank del CSV sono calcolati sull'universo di geni del notebook Python
  # (filtro "somma conteggi >= 3"), diverso da quello di questo script
  # ("espresso in >= 3 spot"). Rank su universi diversi non sono confrontabili
  # in un boxplot, quindi restringo all'intersezione e ricalcolo.
  n_pre_sde <- nrow(sde_orig)
  sde_orig  <- sde_orig %>% filter(gene_id %in% rownames(spe))
  cat("Geni nella tabella SpatialDE:", n_pre_sde, "\n")
  cat("Ristretti all'universo di questo script:", nrow(sde_orig), "su", nrow(spe), "\n")
  cat("  (persi dal CSV:", n_pre_sde - nrow(sde_orig),
      " | assenti dal CSV ma presenti qui:", nrow(spe) - nrow(sde_orig), ")\n")
 
  sde_orig <- sde_orig %>% arrange(qval, desc(LLR)) %>%
    distinct(gene_id, .keep_all = TRUE) %>%        # simboli duplicati dopo la conversione
    mutate(rank = row_number())
  cat("Geni con FDR <", ALPHA_FDR, ":", sum(sde_orig$qval < ALPHA_FDR, na.rm = TRUE), "\n")
 
  # L'effetto e' FSV se presente nel CSV, altrimenti LLR (come faceva la v2).
  eff_sde <- if ("FSV" %in% names(sde_orig)) sde_orig$FSV else sde_orig$LLR
 
  tab_sde <- data.frame(
    gene_id = sde_orig$gene_id, cor = eff_sde, fdr = sde_orig$qval,
    rank = sde_orig$rank, Method = "SpatialDE_orig", tested = TRUE, stringsAsFactors = FALSE)
 
  # Geni dell'universo che SpatialDE non ha nel CSV: come per SPATA2 (11b), vanno
  # in coda, in ordine casuale e con fdr = NA: l'AUC e' calcolata sullo stesso
  # universo di tutti gli altri metodi. I rank dei geni testati non cambiano.
  stopifnot(SDE_ORIG_NON_TESTATI %in% c("coda", "escludi"))
  mancanti_sde <- setdiff(rownames(spe), sde_orig$gene_id)
  cat("Geni dell'universo non presenti nel CSV:", length(mancanti_sde),
      "-> trattamento:", SDE_ORIG_NON_TESTATI, "\n")
  if (SDE_ORIG_NON_TESTATI == "coda" && length(mancanti_sde) > 0) {
    set.seed(SDE_ORIG_SEED_CODA)
    tab_sde <- rbind(tab_sde, data.frame(
      gene_id = mancanti_sde, cor = NA_real_, fdr = NA_real_,
      rank = nrow(sde_orig) + sample.int(length(mancanti_sde)),
      Method = "SpatialDE_orig", tested = FALSE, stringsAsFactors = FALSE))
  }
  results$SpatialDE_orig <- tab_sde
} else {
  sezione_inizio("10b. SpatialDE originale SALTATO (file non trovato)")
}

## 10c. HYBRID - selezione della forma per gene

In [ ]:
# ##############################################################################
# ##   10c. HYBRID - selezione della forma per gene                           ##
# ##                                                                          ##
# ##   Hybrid_GLM     : sceglie fra GLM lineare e GLM gaussiano via BIC.      ##
# ##                    Modelli ANNIDATI, stessi dati (conteggi grezzi),      ##
# ##                    stessa verosimiglianza (Binomiale Negativa): il BIC   ##
# ##                    qui e' un criterio pulito. Logica dello step 14b      ##
# ##                    di pipeline_v2.R, invariata.                          ##
# ##############################################################################
 
if (RUN_HYBRID) {
 
sezione_inizio("10c. Hybrid")
 
universo  <- rownames(logcounts_raw)
n_sp_hyb  <- ncol(logcounts_raw)
i_lin     <- match(universo, glm_lin$gene_id)
i_gau     <- match(universo, glm_gau$gene_id)
 
# percentile dell'effetto, per rendere confrontabili effetti di natura diversa
pct <- function(x) rank(x, na.last = "keep") / sum(!is.na(x))
 
# --- 10c.1 Hybrid_GLM: BIC fra lineare e gaussiano -----------------------------
 
cat("\n-- Hybrid_GLM (lineare vs gaussiano, selezione BIC) --\n")
 
hyg <- data.frame(gene_id = universo, stringsAsFactors = FALSE)
 
bic_lin <- glm_lin$deviance[i_lin] + glm_lin$n_par[i_lin] * log(n_sp_hyb)
bic_gau <- glm_gau$deviance[i_gau] + glm_gau$n_par[i_gau] * log(n_sp_hyb)
hyg$bic_linear <- bic_lin; hyg$bic_gaussian <- bic_gau
 
usa_bic <- !all(is.na(bic_lin)) && !all(is.na(bic_gau))
if (usa_bic) {
  hyg$shape <- ifelse(bic_gau < bic_lin, "gaussian", "linear")
  cat("Selezione della forma: BIC (modelli annidati, criterio pulito)\n")
} else {
  hyg$shape <- ifelse(glm_gau$p_value[i_gau] < glm_lin$p_value[i_lin], "gaussian", "linear")
  cat("ATTENZIONE: devianza non disponibile, selezione ripiegata sul p-value minimo\n")
}
 
p_sel <- ifelse(hyg$shape == "gaussian", glm_gau$p_value[i_gau], glm_lin$p_value[i_lin])
hyg$p_hybrid <- pmin(2 * p_sel, 1)            # Bonferroni: 2 forme provate
hyg$log_p_sel <- ifelse(hyg$shape == "gaussian", glm_gau$log_p[i_gau], glm_lin$log_p[i_lin])
hyg$fdr      <- p.adjust(hyg$p_hybrid, method = "fdr")
 
cat("Geni con FDR <", ALPHA_FDR, ":", sum(hyg$fdr < ALPHA_FDR, na.rm = TRUE), "\n")
cat("Forma selezionata (tutti i geni):\n");            print(table(hyg$shape, useNA = "ifany"))
cat("Forma selezionata (solo geni significativi):\n"); print(table(hyg$shape[hyg$fdr < ALPHA_FDR], useNA = "ifany"))
 
fdr_naive_glm <- p.adjust(pmin(glm_lin$p_value[i_lin], glm_gau$p_value[i_gau]), method = "fdr")
cat("Significativi SENZA correzione per selezione:", sum(fdr_naive_glm < ALPHA_FDR, na.rm = TRUE), "\n")
cat("Significativi CON Bonferroni su 2 modelli   :", sum(hyg$fdr < ALPHA_FDR, na.rm = TRUE), "\n")
 
eff_lin_g  <- abs(glm_lin$coef_dscaled)[i_lin]
eff_gau_g  <- glm_gau_effect[i_gau]
hyg$effect <- ifelse(hyg$shape == "gaussian", pct(eff_gau_g), pct(eff_lin_g))
 
results$Hybrid_GLM <- data.frame(
  gene_id = hyg$gene_id, cor = hyg$effect, fdr = hyg$fdr,
  rank = rank_by_fdr_then_stat(hyg$fdr, hyg$log_p_sel),
  Method = "Hybrid_GLM", stringsAsFactors = FALSE)
 
write_csv(hyg, file.path(OUTPUT_DIR, "hybrid_glm_forme_selezionate.csv"))
 
} else {
  sezione_inizio("10c. Hybrid SALTATO (RUN_HYBRID = FALSE)")
}

## 10d. MORAN'S I E LAPLACIAN SCORE - forma standard

In [ ]:
# ##############################################################################
# ##   10d. MORAN'S I E LAPLACIAN SCORE - forma standard                      ##
# ##                                                                          ##
# ##   Due riferimenti 2D center-free accanto a SpatialDE originale: misurano ##
# ##   quanto l'espressione di un gene e' simile fra spot vicini, senza       ##
# ##   centro e senza covariata. Nessuno dei due usa niente di costruito      ##
# ##   dalla pipeline per i metodi radiali.                                   ##
# ##                                                                          ##
# ##   INPUT   logcounts: log-conteggi                                        ##
# ##           normalizzati per library size, come squidpy/Seurat/Voyager.    ##
# ##           Moran e Laplacian sono invarianti alla scala, quindi log2 o    ##
# ##           ln non cambia nulla.                                           ##
# ##   GRAFO   adiacenza vera della griglia esagonale Visium: i 6 vicini di   ##
# ##           (riga, colonna) sono (r, c+-2) e (r+-1, c+-1), come squidpy    ##
# ##           spatial_neighbors(coord_type = "grid", n_rings = 1). Ai bordi  ##
# ##           il grado scende sotto 6; nessun salto a spot a due passi.      ##
# ##   MORAN   I = (n / S0) * z'Wz / z'z, z centrata sulla media semplice.     ##
# ##           Pesi standardizzati per riga (default spdep e squidpy),        ##
# ##           varianza sotto normalita' (Cliff & Ord), test a una coda       ##
# ##           I > E[I] (come pval_norm di squidpy), BH. Spot isolati: n      ##
# ##           ridotto del loro numero, come spdep con adjust.n = TRUE.       ##
# ##           Rank: FDR, poi Z_I. Il p e' pnorm(Z_I): stesso ordine del p    ##
# ##           esatto anche dove l'FDR pareggia; I da solo non lo e'.         ##
# ##   LAPLACIAN  He, Cai & Niyogi (2005). L = D - W con W binaria            ##
# ##           simmetrica, D = diag(grado). La feature e' centrata sulla      ##
# ##           media PESATA PER IL GRADO, f~ = f - (f'D1 / 1'D1) 1, e lo      ##
# ##           score e' f~'L f~ / f~'D f~ (piu' basso = piu' liscio). Il      ##
# ##           lavoro originale costruisce il grafo nello spazio delle        ##
# ##           feature; qui il grafo e' spaziale (adattamento dichiarato).    ##
# ##           Pesi 0/1: la variante "simple-minded" prevista dal paper.      ##
# ##           Nessun p-value: entra nei rank (boxplot, ROC, GSEA), non nei   ##
# ##           conteggi di geni significativi (nel riepilogo: NA).            ##
# ##                                                                          ##
# ##   Con lo stesso grafo, il numeratore del Laplacian Score e' la somma     ##
# ##   sugli archi di (f_i - f_j)^2, cioe' una forma di Geary's C: i due      ##
# ##   riferimenti saranno molto correlati, non sono due controlli            ##
# ##   indipendenti.                                                          ##
# ##############################################################################
 
sezione_inizio("10d. Moran's I e Laplacian Score")
 
stopifnot(REF2D_MORAN_STYLE %in% c("W", "B"))
 
X_ref2d <- logcounts_raw
stopifnot(identical(colnames(X_ref2d), spots$Barcode))
n_sp <- ncol(X_ref2d)
cat("Input: logcounts (logNormCounts, forma standard)",
    "|", nrow(X_ref2d), "geni x", n_sp, "spot\n")
 
 
# --- 10d.1  Grafo di adiacenza -------------------------------------------------
# Con le coordinate di array (Visium) il grafo e' esatto. Senza (DATA_SOURCE =
# "rdata") si ripiega sui 6-NN in pixel, tenendo solo i vicini entro 1.2 volte
# il passo della griglia: all'interno sono gli stessi 6, ai bordi non si
# aggiungono spot lontani.
 
if (all(c("array_row", "array_col") %in% names(spots)) &&
    !anyNA(spots$array_row) && !anyNA(spots$array_col)) {
 
  ar  <- as.integer(spots$array_row)
  ac  <- as.integer(spots$array_col)
  key <- paste(ar, ac, sep = "_")
  ii  <- integer(0); jj <- integer(0)
  for (o in list(c(0, 2), c(0, -2), c(1, 1), c(1, -1), c(-1, 1), c(-1, -1))) {
    j  <- match(paste(ar + o[1], ac + o[2], sep = "_"), key)
    ok <- !is.na(j)
    ii <- c(ii, which(ok)); jj <- c(jj, j[ok])
  }
  GRAFO_TIPO <- "griglia Visium (array_row/array_col, 1 anello)"
  rm(ar, ac, key)
 
} else {
 
  xy    <- cbind(spots$pixel_col, spots$pixel_row)
  knn_r <- FNN::get.knn(xy, k = 6)
  passo <- median(knn_r$nn.dist[, 1])
  tieni <- as.vector(t(knn_r$nn.dist)) <= 1.2 * passo
  ii    <- rep(seq_len(n_sp), each = 6)[tieni]
  jj    <- as.vector(t(knn_r$nn.index))[tieni]
  GRAFO_TIPO <- "6-NN in pixel entro 1.2 x passo della griglia (coordinate di array assenti)"
  rm(xy, knn_r, passo, tieni)
}
 
W_bin <- sparseMatrix(i = ii, j = jj, x = 1, dims = c(n_sp, n_sp))
W_bin <- W_bin + t(W_bin)          # simmetrica (con la griglia lo e' gia')
W_bin@x[] <- 1                     # binaria
rm(ii, jj)
 
deg   <- as.numeric(rowSums(W_bin))
n_iso <- sum(deg == 0)
cat("Grafo:", GRAFO_TIPO, "\n")
cat("  grado: mediana", median(deg), "| min", min(deg), "| max", max(deg),
    "| spot con 6 vicini", round(100 * mean(deg == 6), 1), "% | spot isolati", n_iso, "\n")
 
 
# --- 10d.2  Moran's I ----------------------------------------------------------
 
W_mor <- if (REF2D_MORAN_STYLE == "W") {
  Diagonal(x = ifelse(deg > 0, 1 / deg, 0)) %*% W_bin     # righe a somma 1 (isolati: 0)
} else W_bin
 
n_m <- n_sp - n_iso                                       # adjust.n = TRUE, come spdep
S0m <- sum(W_mor)
S1m <- 0.5 * sum((W_mor + t(W_mor))^2)
S2m <- sum((rowSums(W_mor) + colSums(W_mor))^2)
 
Y_c     <- X_ref2d - rowMeans(X_ref2d)                    # z = x - media (definizione di Moran)
Moran_I <- (n_m / S0m) * as.numeric(rowSums((Y_c %*% W_mor) * Y_c)) /
           pmax(as.numeric(rowSums(Y_c^2)), 1e-12)
names(Moran_I) <- rownames(X_ref2d)
rm(Y_c); invisible(gc())
 
E_I       <- -1 / (n_m - 1)
Var_I     <- (n_m^2 * S1m - n_m * S2m + 3 * S0m^2) / (S0m^2 * (n_m^2 - 1)) - E_I^2
Z_I       <- (Moran_I - E_I) / sqrt(Var_I)
p_moran   <- pnorm(Z_I, lower.tail = FALSE)              # una coda: I > E[I]
fdr_moran <- p.adjust(p_moran, method = "BH")
 
cat("Moran's I (pesi", REF2D_MORAN_STYLE, "): geni con FDR <", ALPHA_FDR, ":",
    sum(fdr_moran < ALPHA_FDR, na.rm = TRUE), "\n")
 
 
# --- 10d.3  Controllo contro spdep ---------------------------------------------
# Stesso grafo passato a spdep come oggetto nb, stesso stile di pesi. I valori
# di I e dello z-score devono coincidere a meno dell'arrotondamento: e' la prova
# che la formula qui sopra e' quella del pacchetto di riferimento.
 
moran_check <- NULL
if (REF2D_CHECK_SPDEP) {
  if (requireNamespace("spdep", quietly = TRUE)) {
    s_w <- summary(W_bin)
    nb  <- split(as.integer(s_w$j), factor(s_w$i, levels = seq_len(n_sp)))
    nb  <- lapply(nb, function(v) if (length(v) == 0) 0L else sort(v))
    names(nb) <- NULL
    class(nb) <- "nb"
    attr(nb, "region.id") <- as.character(seq_len(n_sp))
    attr(nb, "sym")       <- TRUE
    lw <- spdep::nb2listw(nb, style = REF2D_MORAN_STYLE, zero.policy = TRUE)
 
    set.seed(1)
    espressi <- which(rowSums(X_ref2d > 0) >= 0.1 * n_sp)
    g_chk    <- espressi[sample.int(length(espressi), min(5, length(espressi)))]
    ref_chk  <- t(sapply(g_chk, function(g) {
      mt <- spdep::moran.test(X_ref2d[g, ], lw, randomisation = FALSE,
                              alternative = "greater", zero.policy = TRUE, adjust.n = TRUE)
      c(I = unname(mt$estimate[1]), Z = unname(mt$statistic))
    }))
    moran_check <- data.frame(gene_id    = rownames(X_ref2d)[g_chk],
                              I_pipeline = Moran_I[g_chk], I_spdep = ref_chk[, "I"],
                              Z_pipeline = Z_I[g_chk],     Z_spdep = ref_chk[, "Z"],
                              row.names = NULL)
    diff_max <- max(abs(moran_check$I_pipeline - moran_check$I_spdep),
                    abs(moran_check$Z_pipeline - moran_check$Z_spdep))
    cat("Controllo contro spdep::moran.test (5 geni): differenza massima =",
        signif(diff_max, 3), if (diff_max < 1e-6) "-> OK\n" else "-> ATTENZIONE: non coincide\n")
    print(moran_check, row.names = FALSE)
    rm(s_w, nb, lw)
  } else {
    cat("spdep non installato: controllo saltato (install.packages(\"spdep\")).\n")
  }
}
 
 
# --- 10d.4  Laplacian Score ------------------------------------------------------
 
mu_D  <- as.numeric(X_ref2d %*% deg) / sum(deg)          # media pesata per il grado
Y_D   <- X_ref2d - mu_D
L_mat <- Diagonal(x = deg) - W_bin
 
Lap_score  <- as.numeric(rowSums((Y_D %*% L_mat) * Y_D)) /
              pmax(as.numeric((Y_D^2) %*% deg), 1e-12)
Lap_effect <- -Lap_score                                  # piu' alto = piu' liscio
rm(Y_D, L_mat, mu_D); invisible(gc())
 
cat("Laplacian Score: solo rank (nessun p-value)\n")
cat("Spearman fra Moran's I e -Laplacian Score:",
    round(cor(Moran_I, Lap_effect, method = "spearman"), 3), "\n")
 
 
# --- 10d.5  In tabella risultati -------------------------------------------------
 
results$Moran_I <- data.frame(
  gene_id = rownames(X_ref2d), cor = as.numeric(Moran_I), fdr = fdr_moran,
  rank = order(order(fdr_moran, -as.numeric(Z_I))),     # Z_I: monotono nel p
  Method = "Moran_I", stringsAsFactors = FALSE)
 
results$Laplacian_Score <- data.frame(
  gene_id = rownames(X_ref2d), cor = Lap_effect, fdr = NA_real_,
  rank = order(order(-Lap_effect)),
  Method = "Laplacian_Score", stringsAsFactors = FALSE)
 
# Metodi che non producono p-value: nel riepilogo, nell'ORA e nel Jaccard
# compaiono come NA ("non applicabile"), non come 0 ("nessun gene trovato").
METODI_SENZA_PVALUE <- c("Laplacian_Score")
 
write_csv(data.frame(gene_id = rownames(X_ref2d), Moran_I = as.numeric(Moran_I),
                     Moran_Z = as.numeric(Z_I), Moran_p = p_moran, Moran_fdr = fdr_moran,
                     Laplacian_Score = Lap_score),
          file.path(OUTPUT_DIR, "riferimenti_2D_moran_laplacian.csv"))
 
REF2D_INFO <- data.frame(input = "logcounts", grafo = GRAFO_TIPO,
                         pesi_moran = REF2D_MORAN_STYLE,
                         grado_mediano = median(deg), spot_isolati = n_iso,
                         n_moran = n_m, stringsAsFactors = FALSE)
 
rm(W_mor, X_ref2d); invisible(gc())

## 10e. CALIBRAZIONE SOTTO IPOTESI NULLA PER LA PROFONDITA'

In [ ]:
# ##############################################################################
# ##   10e. CALIBRAZIONE SOTTO IPOTESI NULLA PER LA PROFONDITA'               ##
# ##                                                                          ##
# ##   Domanda: l'offset di libreria del GLM basta a neutralizzare il legame  ##
# ##   fra la covariata (distanza o pseudotime) e la profondita' di          ##
# ##   sequenziamento? Si mescola la covariata fra spot DENTRO strati di     ##
# ##   profondita' simile: il legame reale gene-covariata si rompe, quello   ##
# ##   aggregato covariata-profondita' resta. Se il test e' ben calibrato,   ##
# ##   sui dati mescolati circa il 5% dei geni ha p < 0.05.                  ##
# ##                                                                          ##
# ##   Si usa lo score test della sezione 9c (fit nullo una volta sola +     ##
# ##   prodotti di matrici): nessun GLM viene rifittato. La parte lenta del  ##
# ##   fit nullo (estimateDisp) e' gia' in dge_null se la 9c e' girata;      ##
# ##   altrimenti viene fatta qui una volta.                                 ##
# ##                                                                          ##
# ##   E' una DIAGNOSTICA uguale per tutti i dataset: non modifica nessun    ##
# ##   risultato a valle, produce la tabella di calibrazione (14c, CSV).     ##
# ##############################################################################
 
RUN_CALIBRATION      <- TRUE
N_PERM_DEPTH         <- 500        # 100 per un run di prova
N_STRATA_DEPTH       <- 10
CALIB_SEED           <- 20260926
CALIB_MAX_WORKERS    <- 16         # operazioni limitate dalla memoria: oltre ~16 non accelera
 
if (RUN_CALIBRATION) {
 
sezione_inizio("10e. Calibrazione sotto ipotesi nulla (profondita')")
 
# --- 10e.1  Residui e pesi del fit nullo (intercetta + offset) ----------------
t0 <- Sys.time()
design_null <- matrix(1, nrow = ncol(spe), ncol = 1,
                      dimnames = list(colnames(spe), "(Intercept)"))
if (!exists("dge_null")) {
  cat("   dge_null assente (9c non eseguita): estimateDisp una volta\n")
  dge_null <- estimateDisp(dge, design_null)
  phi_gene <- if (!is.null(dge_null$tagwise.dispersion)) dge_null$tagwise.dispersion else
              rep(dge_null$common.dispersion, nrow(dge_null))
}
fit_null_c <- glmFit(dge_null, design_null)
mu_c       <- fit_null_c$fitted.values
denom_c    <- 1 + phi_gene * mu_c
R_res      <- (as.matrix(counts(spe)) - mu_c) / denom_c
W_wts      <- mu_c / denom_c
rm(fit_null_c, mu_c, denom_c); invisible(gc())
cat("   residui e pesi: ", nrow(R_res), " geni x ", ncol(R_res), " spot (",
    round(as.numeric(difftime(Sys.time(), t0, units = "secs")), 1), " s)\n", sep = "")
 
# --- 10e.2  Score test per una covariata (forma lineare e gaussiana) ----------
score_test_covariata <- function(x_sub, R_sub, W_sub) {
  x1 <- x_sub / max(x_sub)
  x2 <- x1^2
  S0 <- rowSums(W_sub)
  U1 <- as.numeric(R_sub %*% x1);   U2 <- as.numeric(R_sub %*% x2)
  S1 <- as.numeric(W_sub %*% x1);   S2 <- as.numeric(W_sub %*% x2)
  S3 <- as.numeric(W_sub %*% x1^3); S4 <- as.numeric(W_sub %*% x1^4)
  V1 <- S2 - S1^2 / S0
  Z1 <- U1 / sqrt(pmax(V1, 1e-12))
  a11 <- V1; a12 <- S3 - S1 * S2 / S0; a22 <- S4 - S2^2 / S0
  det2 <- a11 * a22 - a12^2
  X2 <- (a22 * U1^2 - 2 * a12 * U1 * U2 + a11 * U2^2) / pmax(det2, 1e-12)
  list(Z1 = Z1, X2 = X2)
}
 
# sample(x) con x di lunghezza 1 e x >= 1 NON restituisce x: campiona da 1:x.
resample_sicuro <- function(x) if (length(x) <= 1) x else sample(x)
 
COVARIATE_DEPTH <- list(
  "Distanza dal centro" = r_center,
  "Gaston"    = spe$isodepth,
  "SpaceFlow" = spe$spaceflow_pt,
  "Slingshot" = spe$slingshot_pt,
  "Monocle"   = spe$monocle_pt
)
 
# --- 10e.3  Permutazioni in parallelo (BLAS a 1 thread nei worker) ------------
nw_perm <- min(CALIB_MAX_WORKERS, N_CORES)
set_blas_threads(1)                       # i worker FORK ereditano 1 thread
cl <- makeCluster(nw_perm, type = "FORK")
registerDoParallel(cl)
cat("   worker:", nw_perm, "x 1 thread BLAS\n")
 
set.seed(CALIB_SEED)
calib_list <- list()
 
for (nome in names(COVARIATE_DEPTH)) {
 
  x_full <- COVARIATE_DEPTH[[nome]]
  if (is.null(x_full)) { cat("  ", nome, ": covariata assente, saltata\n", sep = ""); next }
 
  ok    <- is.finite(x_full)              # esclude NA e Inf (Monocle usa Inf)
  n_inf <- sum(is.infinite(x_full))
  if (n_inf > 0) cat("  ", nome, ": ", n_inf, "/", length(x_full),
                     " spot con Inf esclusi\n", sep = "")
  if (sum(ok) < GLM_PT_MIN_SPOTS) { cat("  ", nome, ": troppi pochi spot finiti, saltata\n", sep = ""); next }
 
  R_sub    <- R_res[, ok, drop = FALSE]
  W_sub    <- W_wts[, ok, drop = FALSE]
  x_ok     <- x_full[ok]
  depth_ok <- spe$sum_umi[ok]
 
  oss <- score_test_covariata(x_ok, R_sub, W_sub)
  frac_lin_oss <- mean(2 * pnorm(-abs(oss$Z1)) < 0.05, na.rm = TRUE)
  frac_gau_oss <- mean(pchisq(oss$X2, df = 2, lower.tail = FALSE) < 0.05, na.rm = TRUE)
 
  strata_idx <- split(seq_along(depth_ok), dplyr::ntile(depth_ok, N_STRATA_DEPTH))
 
  t0 <- Sys.time()
  stampa("  ", nome, ": ", N_PERM_DEPTH, " permutazioni su ", nw_perm,
         " worker (avanzamento delle singole permutazioni solo in avanzamento.txt)")
  perm_res <- foreach(b = seq_len(N_PERM_DEPTH), .combine = rbind) %dopar% {
    x_perm <- x_ok
    for (idx in strata_idx) x_perm[idx] <- resample_sicuro(x_ok[idx])
    r <- score_test_covariata(x_perm, R_sub, W_sub)
    # solo una riga nel file ogni 50 permutazioni: nessun effetto sul calcolo
    if (b %% 50 == 0)
      cat(format(Sys.time(), "%Y-%m-%d %H:%M:%S"), "   10e ", nome, ": permutazione ", b, "/",
          N_PERM_DEPTH, " fatta\n", sep = "", file = LOG_AVANZAMENTO, append = TRUE)
    c(lin = mean(2 * pnorm(-abs(r$Z1)) < 0.05, na.rm = TRUE),
      gau = mean(pchisq(r$X2, df = 2, lower.tail = FALSE) < 0.05, na.rm = TRUE))
  }
 
  frac_lin_perm <- perm_res[, "lin"]
  frac_gau_perm <- perm_res[, "gau"]
  n_nan <- sum(!is.finite(frac_lin_perm)) + sum(!is.finite(frac_gau_perm))
  if (n_nan > 0) cat("  ATTENZIONE - ", nome, ": ", n_nan, " repliche non finite escluse\n", sep = "")
  frac_lin_perm <- frac_lin_perm[is.finite(frac_lin_perm)]
  frac_gau_perm <- frac_gau_perm[is.finite(frac_gau_perm)]
 
  calib_list[[nome]] <- data.frame(
    Covariata               = nome,
    n_spot                  = sum(ok),
    n_inf_esclusi           = n_inf,
    cor_profondita          = round(cor(x_ok, depth_ok, method = "spearman"), 3),
    frac_p05_lin_osservato  = round(frac_lin_oss, 4),
    frac_p05_lin_null_media = round(mean(frac_lin_perm), 4),
    frac_p05_lin_null_p025  = round(quantile(frac_lin_perm, 0.025, names = FALSE), 4),
    frac_p05_lin_null_p975  = round(quantile(frac_lin_perm, 0.975, names = FALSE), 4),
    frac_p05_gau_osservato  = round(frac_gau_oss, 4),
    frac_p05_gau_null_media = round(mean(frac_gau_perm), 4),
    frac_p05_gau_null_p025  = round(quantile(frac_gau_perm, 0.025, names = FALSE), 4),
    frac_p05_gau_null_p975  = round(quantile(frac_gau_perm, 0.975, names = FALSE), 4),
    stringsAsFactors = FALSE)
 
  stampa(sprintf("  %-20s (%.0f s)  lineare: oss=%.3f null=%.3f [%.3f-%.3f]  |  gaussiana: oss=%.3f null=%.3f [%.3f-%.3f]",
              nome, as.numeric(difftime(Sys.time(), t0, units = "secs")),
              frac_lin_oss, mean(frac_lin_perm),
              quantile(frac_lin_perm, 0.025), quantile(frac_lin_perm, 0.975),
              frac_gau_oss, mean(frac_gau_perm),
              quantile(frac_gau_perm, 0.025), quantile(frac_gau_perm, 0.975)))
}
 
stopCluster(cl)
set_blas_threads(N_CORES)                 # si torna al tetto per il processo principale
 
calib_tab <- bind_rows(calib_list)
cat("\n>>> CALIBRAZIONE SOTTO IPOTESI NULLA (atteso: frac_p05_null ~ 0.05) <<<\n")
print(calib_tab, row.names = FALSE)
cat("\nLettura: frac_p05_null vicino a 0.05 = test calibrato. Molto sopra 0.05 =\n")
cat("p-value troppo ottimisti per quella covariata su questo dataset. Se\n")
cat("l'eccesso cresce con |cor_profondita| la causa e' la profondita'.\n")
cat("ATTENZIONE: questa permutazione rende la covariata spazialmente casuale, quindi\n")
cat("NON puo' rilevare l'eccesso dovuto all'autocorrelazione spaziale (spot vicini\n")
cat("non indipendenti). Quello si guarda nella sezione 10f (nullo spaziale).\n")
write_csv(calib_tab, file.path(OUTPUT_DIR, "calibrazione_profondita_score_test.csv"))
 
rm(R_res, W_wts); invisible(gc())
 
} else {
  sezione_inizio("10e. Calibrazione SALTATA (RUN_CALIBRATION = FALSE)")
  calib_tab <- data.frame()
}

## 10f. NULLO SPAZIALE PER I GLM RADIALI: OGNI SPOT COME CENTRO

In [ ]:
# ##############################################################################
# ##   10f. NULLO SPAZIALE PER I GLM RADIALI: OGNI SPOT COME CENTRO            ##
# ##                                                                          ##
# ##   La 10e permuta la covariata fra spot e cosi' la rende spazialmente     ##
# ##   casuale: per costruzione non puo' vedere il problema principale di un  ##
# ##   GLM su dati spaziali. Una covariata liscia (il raggio) messa contro    ##
# ##   un'espressione spazialmente autocorrelata (l'architettura del tessuto) ##
# ##   produce geni "significativi" anche senza nessun centro vero, perche'   ##
# ##   gli spot vicini non sono osservazioni indipendenti.                    ##
# ##                                                                          ##
# ##   Il riferimento giusto e' la stessa statistica calcolata prendendo come ##
# ##   centro ciascuno degli altri spot. E' gia' tutta in Z1_scan / X2_scan   ##
# ##   (sezione 9c): nessun fit nuovo. Per ogni centro candidato si conta     ##
# ##   quanti geni hanno BH-FDR < ALPHA_FDR allo score test, in forma         ##
# ##   lineare e gaussiana, e si guarda dove cade il barcode di partenza.     ##
# ##   Si riporta anche il confronto con i soli centri "lontani" (oltre la    ##
# ##   mediana della distanza dal barcode): non condividono il gradiente del  ##
# ##   centro vero, quindi sono il confronto piu' pulito.                    ##
# ##                                                                          ##
# ##   I conteggi dello score test NON sono quelli del glmQLFTest della       ##
# ##   sezione 7 (niente quasi-likelihood ne' shrinkage): sono confrontabili  ##
# ##   fra centri, non con la colonna n_significativi del riepilogo.          ##
# ##                                                                          ##
# ##   Lettura:                                                               ##
# ##     p_empirico piccolo -> il barcode da' piu' geni significativi di      ##
# ##       quasi tutti gli altri centri: il conteggio dei GLM dice qualcosa.  ##
# ##     p_empirico grande  -> un centro qualunque da' altrettanti geni: il   ##
# ##       numero di significativi dei GLM non e' un argomento a favore del   ##
# ##       metodo o del centro.                                               ##
# ##   Le AUC sul Golden Standard sono basate sui rank e non dipendono dalla  ##
# ##   calibrazione dei p-value: questa diagnostica non le tocca.             ##
# ##############################################################################
 
if (RUN_SPATIAL_NULL && exists("Z1_scan") && exists("X2_scan")) {
 
sezione_inizio("10f. Nullo spaziale: ogni spot come centro (score test)")
 
stopifnot(identical(colnames(Z1_scan), colnames(spe)))
n_cand_null <- ncol(Z1_scan)
 
conta_bh <- function(p) sum(p.adjust(p, method = "BH") < ALPHA_FDR, na.rm = TRUE)
 
t0 <- Sys.time()
nsig_null_lin <- vapply(seq_len(n_cand_null), function(j)
  conta_bh(2 * pnorm(-abs(Z1_scan[, j]))), numeric(1))
nsig_null_gau <- vapply(seq_len(n_cand_null), function(j)
  conta_bh(pchisq(X2_scan[, j], df = 2, lower.tail = FALSE)), numeric(1))
cat("   ", n_cand_null, " centri in ",
    round(as.numeric(difftime(Sys.time(), t0, units = "secs")), 1), " s\n", sep = "")
 
d_da_barcode <- sqrt((spots$pixel_col - spots$pixel_col[center_idx])^2 +
                     (spots$pixel_row - spots$pixel_row[center_idx])^2)
lontani <- d_da_barcode > median(d_da_barcode)
 
riga_null <- function(v, forma) {
  oss <- v[center_idx]
  data.frame(
    Forma              = forma,
    n_sig_barcode      = oss,
    rango_barcode      = rank(-v, ties.method = "min", na.last = "keep")[center_idx],
    n_centri           = sum(!is.na(v)),
    mediana_tutti      = median(v, na.rm = TRUE),
    p95_tutti          = quantile(v, 0.95, na.rm = TRUE, names = FALSE),
    p_empirico_tutti   = mean(v >= oss, na.rm = TRUE),
    mediana_lontani    = median(v[lontani], na.rm = TRUE),
    p95_lontani        = quantile(v[lontani], 0.95, na.rm = TRUE, names = FALSE),
    p_empirico_lontani = (sum(v[lontani] >= oss, na.rm = TRUE) + 1) / (sum(!is.na(v[lontani])) + 1),
    stringsAsFactors = FALSE)
}
 
null_spaziale_tab <- rbind(riga_null(nsig_null_lin, "Lineare"),
                           riga_null(nsig_null_gau, "Gaussiana"))
 
cat("\n>>> NULLO SPAZIALE: GENI SIGNIFICATIVI (score test) PER CENTRO <<<\n")
print(null_spaziale_tab, row.names = FALSE)
cat("\np_empirico_tutti include il barcode stesso; p_empirico_lontani usa la\n")
cat("correzione (k + 1) / (n + 1) perche' il barcode non e' fra i lontani.\n")
 
write_csv(null_spaziale_tab, file.path(OUTPUT_DIR, "nullo_spaziale_centri.csv"))
write_csv(data.frame(barcode = colnames(spe),
                     pixel_col = spots$pixel_col, pixel_row = spots$pixel_row,
                     dist_da_barcode = d_da_barcode, lontano = lontani,
                     n_sig_lineare = nsig_null_lin, n_sig_gaussiana = nsig_null_gau),
          file.path(OUTPUT_DIR, "nullo_spaziale_per_spot.csv"))
 
# --- istogramma: distribuzione sui centri, barcode in rosso ------------------
null_long <- rbind(data.frame(Forma = "Lineare",   n_sig = nsig_null_lin, lontano = lontani),
                   data.frame(Forma = "Gaussiana", n_sig = nsig_null_gau, lontano = lontani))
null_vline <- data.frame(Forma = c("Lineare", "Gaussiana"),
                         n_sig = c(nsig_null_lin[center_idx], nsig_null_gau[center_idx]))
 
p_null <- ggplot(null_long, aes(x = n_sig, fill = ifelse(lontano, "centri lontani", "centri vicini"))) +
  geom_histogram(bins = 50, alpha = 0.75, position = "stack") +
  geom_vline(data = null_vline, aes(xintercept = n_sig), colour = "firebrick", linewidth = 1.1) +
  facet_wrap(~Forma, scales = "free") +
  scale_fill_manual(values = c("centri lontani" = "grey45", "centri vicini" = "steelblue")) +
  theme_minimal(base_size = 13) +
  labs(title = "Nullo spaziale: geni significativi prendendo ogni spot come centro",
       subtitle = paste0(SAMPLE_ID, "  |  linea rossa = CENTER_BARCODE  |  score test, BH < ", ALPHA_FDR),
       x = "Geni con FDR < soglia (score test)", y = "Numero di centri", fill = NULL)
options(repr.plot.width = 11, repr.plot.height = 5)
print(p_null)
ggsave(file.path(PLOT_DIR, "08_nullo_spaziale_istogramma.png"), p_null, width = 11, height = 5, dpi = 150)
 
# --- mappa: dove un centro "qualunque" da' molti geni -----------------------
md_null <- data.frame(x = spots$pixel_col, y = spots$pixel_row, n_sig = nsig_null_lin)
p_null_map <- ggplot(md_null, aes(x = x, y = y, fill = n_sig)) +
  geom_point(shape = 21, size = 2.6, stroke = 0.15) +
  geom_point(data = md_null[center_idx, ], shape = 4, size = 4, stroke = 1.4, colour = "white") +
  scale_fill_viridis_c() + scale_y_reverse() + theme_minimal() +
  labs(title = "Geni significativi (forma lineare) per centro candidato",
       subtitle = paste0(SAMPLE_ID, "  |  x bianca = CENTER_BARCODE"),
       x = "Pixel colonna", y = "Pixel riga", fill = "n geni")
options(repr.plot.width = 7, repr.plot.height = 6)
print(p_null_map)
ggsave(file.path(PLOT_DIR, "08b_nullo_spaziale_mappa.png"), p_null_map, width = 7, height = 6, dpi = 150)
 
rm(null_long, md_null); invisible(gc())
 
} else {
  sezione_inizio("10f. Nullo spaziale SALTATO",
                 if (!RUN_SPATIAL_NULL) "(RUN_SPATIAL_NULL = FALSE)" else "(serve la sezione 9c: RUN_INFERRED_CENTER = TRUE)")
  null_spaziale_tab <- data.frame()
}

## 11. TABELLA UNIFICATA

In [ ]:
# ##############################################################################
# ##   11. TABELLA UNIFICATA                                                  ##
# ##                                                                          ##
# ##   Ogni metodo viene assegnato a un Gruppo in base alla covariata che     ##
# ##   usa: "Distanza dal centro" per i 4 principali, "Pseudotime: <fonte>"   ##
# ##   per gli 8 del blocco D. Il Gruppo serve solo a organizzare i grafici   ##
# ##   (facet), non cambia nessun calcolo.                                   ##
# ##############################################################################
 
sezione_inizio("11. Tabella unificata")
 
GROUP_LEVELS <- c("Riferimento 2D", "Distanza dal centro",
                  paste0("Pseudotime: ", c("Gaston", "SpaceFlow", "Slingshot", "Monocle")))
 
method_group <- function(m) {
  g <- rep("Distanza dal centro", length(m))
  for (pt in c("Gaston", "SpaceFlow", "Slingshot", "Monocle")) {
    hit <- grepl(paste0("_", pt, "$"), m)
    g[hit] <- paste0("Pseudotime: ", pt)
  }
  g[m %in% c("SpatialDE_orig", "Moran_I", "Laplacian_Score")] <- "Riferimento 2D"
  g
}
 
all_results <- bind_rows(results) %>%
  left_join(gene_table, by = "gene_id") %>%
  rename(gene = gene_name) %>%
  mutate(sig = !is.na(fdr) & fdr < ALPHA_FDR, tested = dplyr::coalesce(tested, TRUE),
         Group = factor(method_group(Method), levels = GROUP_LEVELS))
all_results$Group <- droplevels(all_results$Group)
method_lookup <- all_results %>% distinct(Method, Group)
 
cat("Righe:", nrow(all_results), " (", n_distinct(all_results$gene_id), "geni x",
    length(unique(all_results$Method)), "metodi)\n")
cat("Metodi per gruppo:\n")
print(all_results %>% distinct(Method, Group) %>% count(Group, name = "n_metodi"), row.names = FALSE)

## 11b. SPATA2 (tool esterno, workflow standard dell'articolo)

In [ ]:
# ##############################################################################
# ##   11b. SPATA2 (tool esterno, workflow standard dell'articolo)            ##
# ##                                                                          ##
# ##   Parte dagli output Space Ranger originali (SPATA2_VISIUM_DIR) con QC e ##
# ##   normalizzazione sue: non usa niente della pipeline. Come nel paper:    ##
# ##   regione di origine = image annotation / cluster degli autori,          ##
# ##   spatialAnnotationScreening() con n_random = 10000, seed 123,           ##
# ##   rm_zero_infl = SPATA2_RM_ZERO_INFL, distance e resolution da           ##
# ##   SPATA2_ARGS (list() = default del tool, si sceglie dal blocco dataset  ##
# ##   con SPATA2_PARAMETRI_PAPER).                                           ##
# ##   Screening in parallelo a blocchi di geni (v5.1, vedi 11b.3).           ##
# ##   Ranking: fdr crescente, a parita' tot_var (statistica del test).       ##
# ##   Geni dell'universo non testati da SPATA2: in coda, ordine casuale.     ##
# ##############################################################################
 
sezione_inizio("11b. SPATA2")
 
spata2_qc <- NULL
 
if (RUN_SPATA2) {
 
  suppressPackageStartupMessages(library(SPATA2))
  map <- purrr::map                                  # SPATA2 chiama map() senza prefisso
 
  SPATA2_GROUP <- "Tool esterno: SPATA2"
  SPATA2_OUT   <- file.path(OUTPUT_DIR, "spata2")
  dir.create(SPATA2_OUT, recursive = TRUE, showWarnings = FALSE)
 
  pos_file <- list.files(file.path(SPATA2_VISIUM_DIR, "spatial"),
                         pattern = "^tissue_positions(_list)?\\.csv$", full.names = TRUE)[1]
  spata2_pos <- read.csv(pos_file, header = FALSE)
  if (!is.numeric(spata2_pos[[2]])) spata2_pos <- read.csv(pos_file, header = TRUE)
  spata2_pos <- spata2_pos[spata2_pos[[2]] == 1, ]          # solo spot in tessuto
 
 
  # --- 11b.1  Regione di origine ----------------------------------------------
 
# Poligoni delle image annotation degli autori (SPATA2::image_annotations,
# campione MCI_LMU) portati nel sistema di coordinate di SPATA2 3.x.
# I vertici sono salvati nello spazio 600 x 600 px dell'immagine lowres di
# Space Ranger (verificato su H&E e Lcn2/Serpina3n: trasformazione identita',
# nessun flip). x_orig/y_orig = coordinate fullres = px_lowres / tissue_lowres_scalef,
# le stesse di pxl_col/pxl_row in tissue_positions_list.csv.
spata2_poligoni_autori <- function(o, visium_dir) {
  e <- new.env()
  utils::data("image_annotations", package = "SPATA2", envir = e)
  ann <- e$image_annotations[[o$campione]]
  if (is.null(ann) || !all(o$ids %in% names(ann)))
    stop("image_annotations$", o$campione, " non contiene: ", paste(setdiff(o$ids, names(ann)), collapse = ", "))
  sfj <- jsonlite::fromJSON(file.path(visium_dir, "spatial", "scalefactors_json.json"))
  lr  <- file.path(visium_dir, "spatial", "tissue_lowres_image.png")
  if (requireNamespace("png", quietly = TRUE) && file.exists(lr)) {
    d <- dim(png::readPNG(lr))
    if (max(d[1:2]) != 600) stop("immagine lowres ", d[2], "x", d[1], " px: i poligoni sono in 600 px")
  }
  lapply(stats::setNames(o$ids, o$ids), function(id) {
    df <- tibble::as_tibble(ann[[id]]@area$outer)
    tibble::tibble(x_orig = df$x / sfj$tissue_lowres_scalef,
                   y_orig = df$y / sfj$tissue_lowres_scalef)
  })
}
 
# punto nel poligono (ray casting), vettoriale sui punti
spata2_in_poligono <- function(px, py, vx, vy) {
  dentro <- rep(FALSE, length(px)); j <- length(vx)
  for (i in seq_along(vx)) {
    attraversa <- ((vy[i] > py) != (vy[j] > py)) &
                  (px < (vx[j] - vx[i]) * (py - vy[i]) / (vy[j] - vy[i]) + vx[i])
    dentro <- xor(dentro, attraversa); j <- i
  }
  dentro
}
 
spata2_origin_barcodes <- function(o, visium_dir) {
  if (o$tipo == "csv") {
    a  <- readr::read_csv(o$path, show_col_types = FALSE)
    bc <- if (is.null(o$colonna_barcode)) "Barcode" else o$colonna_barcode
    return(a[[bc]][!is.na(a[[o$colonna]]) & a[[o$colonna]] == o$valore])
  }
  if (o$tipo == "seurat") {
    e  <- new.env(); nm <- load(o$path, envir = e)
    md <- e[[nm[1]]]@meta.data
    bcs <- rownames(md)[!is.na(md[[o$colonna]]) & md[[o$colonna]] == o$valore]
    if (!is.null(o$suffisso)) bcs <- sub("-[0-9]+$", o$suffisso, bcs)
    return(bcs)
  }
  if (o$tipo == "seurat_ricalcolo") {
    # clustering del solo campione come nel Rmd degli autori/utente
    # (SCTransform -> PCA -> FindNeighbors(dims) -> FindClusters(risoluzione))
    cnt <- Seurat::Read10X(file.path(visium_dir, "filtered_feature_bc_matrix"))
    B2  <- SeuratObject::CreateSeuratObject(counts = cnt, assay = "Spatial")
    pf  <- list.files(file.path(visium_dir, "spatial"), pattern = "^tissue_positions(_list)?\\.csv$", full.names = TRUE)[1]
    pos <- read.csv(pf, header = FALSE); if (!is.numeric(pos[[2]])) pos <- read.csv(pf, header = TRUE)
    B2  <- subset(B2, cells = intersect(colnames(B2), pos[[1]][pos[[2]] == 1]))
    B2  <- subset(B2, features = rownames(B2)[!grepl("(^MT-)|(^mt-)", rownames(B2))])
    m   <- suppressWarnings(SeuratObject::GetAssayData(B2, slot = "counts", assay = "Spatial"))
    if (any(Matrix::rowSums(m) == 0)) B2 <- subset(B2, features = rownames(B2)[Matrix::rowSums(m) > 0])
    m   <- suppressWarnings(SeuratObject::GetAssayData(B2, slot = "counts", assay = "Spatial"))
    if (any(Matrix::colSums(m) == 0)) B2 <- subset(B2, cells = colnames(B2)[Matrix::colSums(m) > 0])
    m   <- suppressWarnings(SeuratObject::GetAssayData(B2, slot = "counts", assay = "Spatial"))
    if (any(Matrix::rowSums(m > 0) < 3)) B2 <- subset(B2, features = rownames(B2)[Matrix::rowSums(m > 0) >= 3])
    B2 <- Seurat::SCTransform(B2, assay = "Spatial", new.assay.name = "SCT", verbose = FALSE)
    SeuratObject::DefaultAssay(B2) <- "SCT"
    B2 <- Seurat::RunPCA(B2, assay = "SCT", verbose = FALSE)
    B2 <- Seurat::FindNeighbors(B2, reduction = "pca", dims = o$dims, verbose = FALSE)
    B2 <- Seurat::FindClusters(B2, resolution = o$risoluzione, verbose = FALSE)
    cat("Cluster ricalcolati (risoluzione", o$risoluzione, "):\n"); print(table(B2$seurat_clusters))
    readr::write_csv(tibble::tibble(barcode = colnames(B2), seurat_cluster = as.character(B2$seurat_clusters)),
                     file.path(SPATA2_OUT, paste0("cluster_ricalcolati_res", o$risoluzione, ".csv")))
    return(colnames(B2)[as.character(B2$seurat_clusters) == as.character(o$valore)])
  }
  if (o$tipo == "annotazione_immagine") {
    # spot in tessuto (secondo il positions della cartella) dentro i poligoni
    polys <- spata2_poligoni_autori(o, visium_dir)
    pf  <- list.files(file.path(visium_dir, "spatial"), pattern = "^tissue_positions(_list)?\\.csv$", full.names = TRUE)[1]
    pos <- read.csv(pf, header = FALSE); if (!is.numeric(pos[[2]])) pos <- read.csv(pf, header = TRUE)
    pos <- pos[pos[[2]] == 1, ]
    dentro <- Reduce(`|`, lapply(polys, function(p) spata2_in_poligono(pos[[6]], pos[[5]], p$x_orig, p$y_orig)))
    return(as.character(pos[[1]][dentro]))
  }
  stop("SPATA2_ORIGIN$tipo deve essere 'csv', 'seurat', 'seurat_ricalcolo' o 'annotazione_immagine'.")
    }
 
 
  # --- 11b.2  Oggetto SPATA2 (workflow standard) ---------------------------------
 
  t0_spata2 <- Sys.time()
  stampa("SPATA2: regione di origine (tipo ", SPATA2_ORIGIN$tipo, ") ...")
  origin_bc <- spata2_origin_barcodes(SPATA2_ORIGIN, SPATA2_VISIUM_DIR)
  stampa("SPATA2: spot della regione di origine: ", length(origin_bc))
  stopifnot(length(origin_bc) > 0)
 
  stampa("SPATA2: creazione dell'oggetto dagli output Space Ranger ...")
 
  img_act <- if (file.exists(file.path(SPATA2_VISIUM_DIR, "spatial", "tissue_lowres_image.png"))) "lowres" else "hires"
  obj <- initiateSpataObjectVisium(sample_name = SAMPLE_ID, directory_visium = SPATA2_VISIUM_DIR,
                                   img_active = img_act, img_ref = "hires", verbose = FALSE)
 
  # simbolo SPATA2 -> Ensembl, nello stesso ordine delle feature di Space Ranger
  rn <- rownames(getCountMatrix(obj))
  h5 <- file.path(SPATA2_VISIUM_DIR, "filtered_feature_bc_matrix.h5")
  ens <- if (file.exists(h5)) {
    rownames(suppressMessages(Seurat::Read10X_h5(h5, use.names = FALSE)))
  } else {
    ft <- readr::read_tsv(list.files(file.path(SPATA2_VISIUM_DIR, "filtered_feature_bc_matrix"),
                                     pattern = "features.tsv(.gz)?$", full.names = TRUE)[1],
                          col_names = FALSE, show_col_types = FALSE)
    ft[[1]][!duplicated(ft[[2]])]
  }
  stopifnot(length(ens) == length(rn))
  gene_map <- tibble::tibble(gene = rn, gene_id = ens)
 
  off <- setdiff(getBarcodes(obj), spata2_pos[[1]])
  if (length(off) > 0) obj <- removeObs(obj, barcodes = off)
  n_spot0 <- nObs(obj)
 
  stampa("SPATA2: QC e normalizzazione (", n_spot0, " spot) ...")
  obj <- identifyTissueOutline(obj)
  obj <- identifySpatialOutliers(obj)
  obj <- removeSpatialOutliers(obj)
  obj <- removeGenesZeroCounts(obj)
  obj <- removeObsZeroCounts(obj)
  obj <- normalizeCounts(obj)                        # LogNormalize, default del tool
 
  bcs <- getBarcodes(obj)
  n_origin_in_obj <- sum(origin_bc %in% bcs)
  stampa("SPATA2: dopo il QC ", length(bcs), " spot, ", length(getGenes(obj)), " geni; spot di origine presenti: ",
         n_origin_in_obj, "/", length(origin_bc))
  stopifnot(n_origin_in_obj > 0)
 
  if (SPATA2_ORIGIN$tipo == "annotazione_immagine") {
    # come nel paper: image annotation disegnate dagli autori
    polys <- spata2_poligoni_autori(SPATA2_ORIGIN, SPATA2_VISIUM_DIR)
    for (id in names(polys))
      obj <- addSpatialAnnotation(obj, tags = "injury", area = list(outer = polys[[id]]), id = id,
                                  class = "ImageAnnotation", parent_name = img_act, overwrite = TRUE)
  } else {
    obj <- addFeatures(obj, feature_df = tibble::tibble(
                         barcodes = bcs, origin = factor(ifelse(bcs %in% origin_bc, "origin", "other"))),
                       overwrite = TRUE)
    obj <- createGroupAnnotations(obj, grouping = "origin", group = "origin", id = "origin",
                                  tags = "origin_region", overwrite = TRUE)
  }
 
 
  # --- 11b.3  Screening: una sola chiamata a spatialAnnotationScreening() -----
  #
  # Sostituisce per intero la vecchia 11b.3: da "# --- 11b.3" fino alla riga
  # prima di "# --- 11b.4" (cancella anche spata2_screening_parallelo()).
  #
  # Perche' la versione a blocchi si bloccava (verificato su stab_inj1):
  # do.call(spatialAnnotationScreening, c(list(obj, ...))) mette l'OGGETTO
  # INTERO dentro la chiamata, e all'interno di SPATA2 quella chiamata viene
  # convertita in testo (deparse): ore di CPU per ogni blocco. Con quote(obj)
  # nella chiamata entra solo il nome "obj". Test sullo stesso oggetto,
  # 1.5mm / 50um, 1 core:
  #   20 geni, quote(obj)        -> 0.97 min
  #   tutti i 18814, quote(obj)  -> 3.24 min
  #   20 geni, come v5 (obj)     -> oltre 10 min fermo nel deparse
  # Quindi niente blocchi e niente processi paralleli: una chiamata, pochi
  # minuti, e l'fdr e' quello calcolato da SPATA2 su tutti i geni.
  #
  # SPATA2 usa str_c() senza prefisso: senza stringr caricato si ferma con
  # "could not find function str_c" (come gia' succedeva con map()).

  suppressPackageStartupMessages(library(stringr))

  t0 <- Sys.time()
  stampa("spatialAnnotationScreening su ", length(getGenes(obj)), " geni, rm_zero_infl = ",
         SPATA2_RM_ZERO_INFL, ", ", if (length(SPATA2_ARGS) == 0) "parametri di default" else
           paste(names(SPATA2_ARGS), unlist(SPATA2_ARGS), sep = " = ", collapse = ", "))
  set_blas_threads(1)
  sas <- do.call(spatialAnnotationScreening,
                 c(list(quote(obj),                       # il NOME, non l'oggetto
                        ids = getSpatAnnIds(obj), variables = getGenes(obj),
                        core = FALSE, rm_zero_infl = SPATA2_RM_ZERO_INFL,
                        verbose = FALSE),   # niente barre di avanzamento
                   SPATA2_ARGS))
  set_blas_threads(N_CORES)
  SPATA2_N_WORKERS <- 1L                                  # per la tabella QC
  saveRDS(sas, file.path(SPATA2_OUT, "spata2_sas_object.rds"))

  spata2_res <- tibble::as_tibble(sas@results$significance) %>%
    dplyr::rename(gene = variables) %>%
    dplyr::left_join(gene_map, by = "gene")
  readr::write_csv(spata2_res, file.path(SPATA2_OUT, "spata2_results.csv"))
  rm(sas, obj); invisible(gc())
  stampa("Screening completato in ", minuti_da(t0), " min (SPATA2 in tutto: ", minuti_da(t0_spata2), " min)")

  # --- 11b.4  Classifica nell'universo della pipeline ----------------------------
 
  universo_sp <- unique(as.character(all_results$gene_id[!grepl("^SPATA2", as.character(all_results$Method))]))
 
  sp <- spata2_res %>%
    dplyr::mutate(key = if (GENE_ID_TYPE == "symbol") gene else gene_id) %>%
    dplyr::filter(!is.na(key), key %in% universo_sp) %>%
    dplyr::arrange(fdr, tot_var) %>%
    dplyr::distinct(key, .keep_all = TRUE)
 
  tab <- data.frame(gene_id = sp$key,
                    cor  = -sp$tot_var,              # piu' alto = gradiente piu' liscio
                    fdr  = sp$fdr,
                    rank = seq_len(nrow(sp)),
                    Method = "SPATA2", tested = !is.na(sp$fdr), stringsAsFactors = FALSE)
 
  mancanti <- setdiff(universo_sp, sp$key)
  if (length(mancanti) > 0) {
    set.seed(SPATA2_SEED_CODA)
    tab <- rbind(tab, data.frame(gene_id = mancanti, cor = NA_real_, fdr = NA_real_,
                                 rank = nrow(sp) + sample.int(length(mancanti)),
                                 Method = "SPATA2", tested = FALSE, stringsAsFactors = FALSE))
  }
  results$SPATA2 <- tab
  ns <- setdiff(universo_sp, sp$key[!is.na(sp$fdr)])
  readr::write_csv(data.frame(gene_id = ns, n_spots_detected = as.integer(rowSums(logcounts_raw[ns, , drop = FALSE] > 0)),
                              in_spata2_results = ns %in% sp$key, stringsAsFactors = FALSE),
                   file.path(SPATA2_OUT, "spata2_non_testati.csv"))
 
 
  # --- 11b.5  Dentro all_results (sostituisce eventuali righe SPATA2) -------------
 
  righe_sp <- results$SPATA2 %>%
    dplyr::left_join(gene_table, by = "gene_id") %>%
    dplyr::rename(gene = gene_name) %>%
    dplyr::mutate(sig = !is.na(fdr) & fdr < ALPHA_FDR, Group = SPATA2_GROUP)
 
  livelli <- unique(c(intersect(GROUP_LEVELS, unique(as.character(all_results$Group))), SPATA2_GROUP))
  all_results <- all_results %>%
    dplyr::mutate(Method = as.character(Method), Group = as.character(Group)) %>%
    dplyr::filter(!grepl("^SPATA2", Method)) %>%
    dplyr::bind_rows(righe_sp) %>%
    dplyr::mutate(Group = factor(Group, levels = livelli))
  GROUP_LEVELS  <- unique(c(GROUP_LEVELS, SPATA2_GROUP))
  method_lookup <- all_results %>% dplyr::distinct(Method, Group)
 
 
  # --- 11b.6  Riepilogo ----------------------------------------------------------
 
  spata2_qc <- data.frame(
    dataset = SAMPLE_ID,
    spata2_versione = as.character(utils::packageVersion("SPATA2")),
    spot_dopo_qc = length(bcs), spot_prima_qc = n_spot0,
    spot_origine = length(origin_bc), spot_origine_in_oggetto = n_origin_in_obj,
    center_barcode_in_origine = colnames(spe)[center_idx] %in% origin_bc,
    rm_zero_infl = SPATA2_RM_ZERO_INFL,
    distance = if (is.null(SPATA2_ARGS$distance)) "default" else SPATA2_ARGS$distance,
    resolution = if (is.null(SPATA2_ARGS$resolution)) "default" else SPATA2_ARGS$resolution,
    geni_testati = nrow(spata2_res), geni_testati_nell_universo = nrow(sp),
    geni_non_testati_in_coda = length(mancanti),
    geni_fdr_0.05 = sum(sp$fdr < 0.05, na.rm = TRUE),
    processi_screening = min(SPATA2_N_WORKERS, N_CORES_DISPONIBILI),
    minuti = minuti_da(t0_spata2),
    stringsAsFactors = FALSE)
  readr::write_csv(spata2_qc, file.path(SPATA2_OUT, "spata2_riepilogo.csv"))
  print(spata2_qc, row.names = FALSE)
 
} else {
  cat("RUN_SPATA2 = FALSE: sezione saltata.\n")
}

## 12. GOLDEN STANDARD  (da pipeline_v2.R step 16, invariato)

In [ ]:
# ##############################################################################
# ##   12. GOLDEN STANDARD  (da pipeline_v2.R step 16, invariato)             ##
# ##                                                                          ##
# ##   Fissa l'ordine dei metodi (migliore -> peggiore) usato da tutti i      ##
# ##   grafici seguenti: mediana del rank sui geni del Golden Standard.       ##
# ##############################################################################
 
sezione_inizio("12. Golden Standard")
 
HAS_GS <- !is.null(GOLDEN_STANDARD_PATH) && .has(GOLDEN_STANDARD_PATH)
 
if (!HAS_GS) {
  cat("GOLDEN_STANDARD_PATH assente: boxplot e ROC saltati, ordine provvisorio alfabetico.\n")
  all_results$is_GS <- FALSE
} else {
  paper <- read_csv(GOLDEN_STANDARD_PATH, show_col_types = FALSE)
  sym_col <- intersect(c("names", "gene", "gene_name", "symbol"), names(paper))[1]
  p_col   <- intersect(c("pvals_adj_anatomy", "pvals_adj", "padj", "p_val_adj"), names(paper))[1]
  stopifnot(!is.na(sym_col))
 
  n_pre <- nrow(paper)
  if (!is.na(GS_ALPHA) && !is.na(p_col)) {
    paper <- paper %>% filter(.data[[p_col]] < GS_ALPHA)
    cat("Golden Standard filtrato su", p_col, "<", GS_ALPHA, ":", n_pre, "->", nrow(paper), "geni\n")
  }
 
  gs_symbols <- unique(tolower(trimws(paper[[sym_col]])))
  all_results$is_GS <- tolower(trimws(all_results$gene)) %in% gs_symbols
 
  n_gs <- length(unique(all_results$gene_id[all_results$is_GS]))
  cat("Geni del Golden Standard trovati nel dataset:", n_gs, "\n")
  cat("Frazione dell'universo:", sprintf("%.1f%%", 100 * n_gs / n_distinct(all_results$gene_id)), "\n")
  if (n_gs / n_distinct(all_results$gene_id) > 0.3)
    cat("ATTENZIONE: il Golden Standard copre oltre il 30% dell'universo, boxplot e ROC saranno poco informativi.\n")
  if (n_gs < 10) { cat("Meno di 10 geni GS nel dataset: boxplot e ROC saltati.\n"); HAS_GS <- FALSE }
}
 
if (HAS_GS) {
  METHOD_ORDER <- all_results %>% filter(is_GS) %>% group_by(Method) %>%
    summarise(m = median(rank, na.rm = TRUE), .groups = "drop") %>%
    arrange(m) %>% pull(Method)
} else {
  METHOD_ORDER <- sort(unique(all_results$Method))   # provvisorio, alfabetico
}
cat("\nOrdine dei metodi (migliore -> peggiore) per tutti i grafici:\n")
cat(paste(METHOD_ORDER, collapse = " > "), "\n")
all_results$Method <- factor(all_results$Method, levels = METHOD_ORDER)

## 12b. CHECKPOINT - SALVATAGGIO PRIMA DEI GRAFICI

In [ ]:
# ##############################################################################
# ##   12b. CHECKPOINT - SALVATAGGIO PRIMA DEI GRAFICI                        ##
# ##                                                                          ##
# ##   A questo punto tutto il lavoro pesante e' finito: caricamento dati,    ##
# ##   GLM, ricerca del centro inferito, pseudotime, Hybrid,                  ##
# ##   tabella unificata e Golden Standard (METHOD_ORDER). Da qui in poi      ##
# ##   (sezioni 13-20) e' solo bootstrap, enrichment (ORA/GSEA) e grafici:   ##
# ##   la parte che si modifica piu' spesso.                                 ##
# ##                                                                          ##
# ##   Si salva l'intero ambiente in un .RData: in un altro chunk/script      ##
# ##   basta un load() per ripartire dalla sezione 13 senza rifare le         ##
# ##   sezioni 1-12 (che sono quelle lente).                                 ##
# ##                                                                          ##
# ##   Uso in un altro chunk:                                                ##
# ##     load("<OUTPUT_DIR>/checkpoint_pre_grafici.RData")                   ##
# ##   e da li' si puo' rilanciare la sezione 13 in poi, modificando i        ##
# ##   grafici quante volte si vuole senza ricalcolare nulla a monte.        ##
# ##                                                                          ##
# ##   In un kernel NUOVO il load() ripristina gli oggetti ma non i          ##
# ##   pacchetti: prima del load() rilancia il blocco library(...) in cima,  ##
# ##   dopo il load() chiama set_blas_threads(N_CORES) per il tetto di 40.   ##
# ##############################################################################
 
CHECKPOINT_PATH <- file.path(OUTPUT_DIR, "checkpoint_pre_grafici.RData")
sezione_inizio("12b. Salvataggio checkpoint pre-grafici:", CHECKPOINT_PATH, "")
save.image(file = CHECKPOINT_PATH)
cat("Checkpoint salvato (", round(file.size(CHECKPOINT_PATH) / 1e6, 1), " MB).\n", sep = "")
cat("Per riprendere da qui senza ricalcolare tutto, in un altro chunk:\n")
cat("  load(\"", CHECKPOINT_PATH, "\")\n", sep = "")

## 13. BOXPLOT DEI RANK SUL GOLDEN STANDARD  (orizzontale)

In [ ]:
# ##############################################################################
# ##   13. BOXPLOT DEI RANK SUL GOLDEN STANDARD  (orizzontale)                ##
# ##                                                                          ##
# ##   Metodi sull'asse y, il migliore (mediana piu' bassa, da METHOD_ORDER)  ##
# ##   in alto. Un pannello per gruppo, impilati, con altezza proporzionale   ##
# ##   al numero di metodi: i box hanno tutti lo stesso spessore. La mediana  ##
# ##   e' scritta a destra di ogni box. L'altezza della figura cresce con il  ##
# ##   numero di metodi.                                                      ##
# ##############################################################################
 
if (HAS_GS) {
  sezione_inizio("13. Boxplot Golden Standard")
 
  gs_data <- all_results %>% filter(is_GS, !is.na(rank)) %>%
    mutate(Method = factor(as.character(Method), levels = rev(as.character(METHOD_ORDER))))
  gs_label <- gs_data %>% group_by(Method, Group) %>%
    summarise(med = round(median(rank, na.rm = TRUE), 0),
              n_gs = n(), .groups = "drop")
 
  n_metodi <- n_distinct(gs_data$Method)
  x_max    <- max(gs_data$rank, na.rm = TRUE)
 
  p_box <- ggplot(gs_data, aes(x = rank, y = Method, fill = Group)) +
    geom_boxplot(outlier.shape = NA, alpha = 0.6, width = 0.65, linewidth = 0.4) +
    geom_jitter(width = 0, height = 0.15, size = 0.5, alpha = 0.15, colour = "grey20") +
    geom_label(data = gs_label, aes(x = x_max * 1.03, y = Method, label = med),
               inherit.aes = FALSE, hjust = 0, fontface = "bold", size = 3.8,
               label.size = 0, fill = "white", colour = "grey10") +
    facet_grid(Group ~ ., scales = "free_y", space = "free_y",
               labeller = labeller(Group = label_wrap_gen(width = 16))) +
    scale_x_continuous(expand = expansion(mult = c(0.01, 0.12))) +
    scale_fill_brewer(palette = "Set2") +
    coord_cartesian(clip = "off") +
    theme_minimal(base_size = 13) +
    theme(axis.text.y  = element_text(face = "bold", size = 11, colour = "grey10"),
          axis.text.x  = element_text(size = 10),
          axis.title.x = element_text(face = "bold"),
          strip.text.y = element_text(face = "bold", size = 11, angle = 0, hjust = 0),
          panel.grid.major.y = element_blank(),
          panel.grid.minor   = element_blank(),
          panel.border  = element_rect(colour = "grey80", fill = NA, linewidth = 0.4),
          panel.spacing.y = unit(0.6, "lines"),
          legend.position = "none") +
    labs(title = "Rank dei geni del Golden Standard",
         subtitle = paste0(SAMPLE_ID, "  |  il numero a destra e' la mediana  |  piu' a sinistra e' meglio"),
         x = "Rank", y = NULL)
 
  box_h <- max(6, 0.38 * n_metodi + 2)
  options(repr.plot.width = 14, repr.plot.height = box_h)
  print(p_box)
  ggsave(file.path(PLOT_DIR, "01_boxplot_golden_standard.png"), p_box,
         width = 14, height = box_h, dpi = 150)
  write_csv(gs_label, file.path(OUTPUT_DIR, "boxplot_golden_standard_summary.csv"))
} else {
  sezione_inizio("13. Boxplot Golden Standard SALTATO")
}

## 14. ROC  (step 18, invariato)

In [ ]:
# ##############################################################################
# ##   14. ROC  (step 18, invariato)                                          ##
# ##############################################################################
 
if (HAS_GS) {
  sezione_inizio("14. ROC")
 
  compute_roc <- function(rank_vec, is_gs_vec) {
    ord <- order(rank_vec); g <- is_gs_vec[ord]
    n1 <- sum(g); n0 <- sum(!g)
    if (n1 == 0 || n0 == 0) return(NULL)
    data.frame(FPR = c(0, cumsum(!g) / n0), TPR = c(0, cumsum(g) / n1))
  }
  compute_auc <- function(fpr, tpr) sum(diff(fpr) * (head(tpr, -1) + tail(tpr, -1)) / 2)
 
  roc_list <- list(); auc_vals <- c()
  method_to_group <- setNames(as.character(method_lookup$Group), method_lookup$Method)
  for (m in METHOD_ORDER) {
    d <- all_results %>% filter(Method == m, !is.na(rank))
    r <- compute_roc(d$rank, d$is_GS)
    if (is.null(r)) next
    auc_vals[m] <- compute_auc(r$FPR, r$TPR)
    roc_list[[m]] <- r %>% mutate(Method = m, Group = method_to_group[[m]])
  }
  roc_df <- bind_rows(roc_list)
  auc_tab <- data.frame(Method = names(auc_vals), AUC = round(auc_vals, 3),
                        Group = method_to_group[names(auc_vals)]) %>%
    arrange(Group, desc(AUC))
  cat("AUC per metodo:\n"); print(auc_tab, row.names = FALSE)
  write_csv(auc_tab, file.path(OUTPUT_DIR, "roc_auc.csv"))
 
  roc_df <- roc_df %>% left_join(auc_tab %>% select(Method, AUC), by = "Method") %>%
    mutate(MethodLabel = sprintf("%s (AUC=%.3f)", Method, AUC))
 
  p_roc <- ggplot(roc_df, aes(x = FPR, y = TPR, colour = MethodLabel)) +
    geom_line(linewidth = 0.9) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", colour = "grey50") +
    facet_wrap(~Group) +
    coord_equal() + theme_minimal() +
    theme(strip.text = element_text(face = "bold"), legend.text = element_text(size = 7),
          legend.position = "bottom") +
    guides(colour = guide_legend(ncol = 2)) +
    labs(title = paste0("ROC - ", n_distinct(roc_df$Method), " metodi in ", n_distinct(roc_df$Group), " gruppi"),
         subtitle = SAMPLE_ID,
         x = "False Positive Rate", y = "True Positive Rate", colour = "Metodo (AUC)")
 
  print(p_roc)
  n_gruppi_roc <- n_distinct(roc_df$Group)
  ggsave(file.path(PLOT_DIR, "02_roc_curve.png"), p_roc,
        width = max(8, 3.5 * min(n_gruppi_roc, 3)),
        height = 5.5 * ceiling(n_gruppi_roc / 3) + 2, dpi = 150)
} else {
  sezione_inizio("14. ROC SALTATO")
}

## 14b. BOOTSTRAP: INTERVALLI DI CONFIDENZA SULLE AUC

In [ ]:
# ##############################################################################
# ##   14b. BOOTSTRAP: INTERVALLI DI CONFIDENZA SULLE AUC                     ##
# ##                                                                          ##
# ##   Per ogni metodo si ricampionano i geni con ripetizione e si ricalcola  ##
# ##   l'AUC N_BOOT volte. L'intervallo e' il 2.5%-97.5% delle repliche       ##
# ##   (bootstrap percentile). Sono intervalli MARGINALI: descrivono         ##
# ##   l'incertezza di ogni AUC, ma la non sovrapposizione fra due metodi    ##
# ##   non e' un test. Nessun confronto appaiato fra metodi.                ##
# ##############################################################################
 
N_BOOT    <- 1000
BOOT_SEED <- 20260926
 
if (HAS_GS) {
  sezione_inizio("14b. Bootstrap sulle AUC (Golden Standard)")
  set.seed(BOOT_SEED)
 
  boot_auc_one_method <- function(d, n_boot) {
    n_d <- nrow(d)
    out <- numeric(n_boot)
    for (b in seq_len(n_boot)) {
      idx <- sample.int(n_d, n_d, replace = TRUE)
      r   <- compute_roc(d$rank[idx], d$is_GS[idx])
      out[b] <- if (is.null(r)) NA_real_ else compute_auc(r$FPR, r$TPR)
    }
    out
  }
 
  auc_ci_list <- list()
  t0_boot <- Sys.time()
  for (m in METHOD_ORDER) {
    d <- all_results %>% filter(Method == m, !is.na(rank))
    if (nrow(d) < 10 || sum(d$is_GS) == 0 || !m %in% names(auc_vals)) next
    stampa("  bootstrap ", match(m, METHOD_ORDER), "/", length(METHOD_ORDER), ": ", m,
           " | trascorsi ", minuti_da(t0_boot), " min")
    bv <- boot_auc_one_method(d, N_BOOT)
    bv <- bv[is.finite(bv)]
    auc_ci_list[[m]] <- data.frame(
      Method  = m,
      AUC     = unname(auc_vals[m]),
      CI_low  = quantile(bv, 0.025, names = FALSE),
      CI_high = quantile(bv, 0.975, names = FALSE))
    cat(sprintf("  %-25s AUC=%.3f  [%.3f, %.3f]\n", m, auc_vals[m],
                auc_ci_list[[m]]$CI_low, auc_ci_list[[m]]$CI_high))
  }
  auc_ci_tab <- bind_rows(auc_ci_list) %>%
    left_join(method_lookup %>% mutate(Method = as.character(Method)), by = "Method") %>%
    arrange(Group, desc(AUC))
  write_csv(auc_ci_tab, file.path(OUTPUT_DIR, "auc_bootstrap_ci.csv"))
 
  p_auc_ci <- ggplot(auc_ci_tab,
                     aes(x = AUC, y = factor(Method, levels = rev(METHOD_ORDER)),
                         xmin = CI_low, xmax = CI_high, colour = Group)) +
    geom_pointrange(size = 0.4) +
    theme_minimal(base_size = 12) +
    theme(axis.text.y = element_text(size = 8)) +
    labs(title = "AUC con intervallo di confidenza bootstrap (95%)",
         subtitle = paste0(SAMPLE_ID, " - ", N_BOOT, " repliche"),
         x = "AUC", y = NULL, colour = NULL)
  options(repr.plot.width = 10, repr.plot.height = 10)
  print(p_auc_ci)
  ggsave(file.path(PLOT_DIR, "02b_auc_bootstrap_ci.png"), p_auc_ci, width = 9, height = 10, dpi = 150)
} else {
  sezione_inizio("14b. Bootstrap SALTATO (nessun Golden Standard)")
  auc_ci_tab <- data.frame()
}

## 14c. GRAFICO DELLA CALIBRAZIONE (tabella calcolata in 10e)

In [ ]:
# ##############################################################################
# ##   14c. GRAFICO DELLA CALIBRAZIONE (tabella calcolata in 10e)             ##
# ##############################################################################
 
if (nrow(calib_tab) > 0) {
  sezione_inizio("14c. Grafico calibrazione")
 
  calib_long <- bind_rows(
    calib_tab %>% transmute(Covariata, Forma = "Lineare",
                            Osservato  = frac_p05_lin_osservato,
                            Null_media = frac_p05_lin_null_media,
                            Null_p025  = frac_p05_lin_null_p025,
                            Null_p975  = frac_p05_lin_null_p975),
    calib_tab %>% transmute(Covariata, Forma = "Gaussiana",
                            Osservato  = frac_p05_gau_osservato,
                            Null_media = frac_p05_gau_null_media,
                            Null_p025  = frac_p05_gau_null_p025,
                            Null_p975  = frac_p05_gau_null_p975))
 
  p_calib <- ggplot(calib_long, aes(x = Covariata)) +
    geom_hline(yintercept = 0.05, linetype = "dashed", colour = "grey40") +
    geom_pointrange(aes(y = Null_media, ymin = Null_p025, ymax = Null_p975),
                    colour = "steelblue", size = 0.5) +
    geom_point(aes(y = Osservato), colour = "firebrick", size = 2.5, shape = 17) +
    facet_wrap(~Forma) +
    theme_minimal(base_size = 13) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
    labs(title = "Calibrazione sotto ipotesi nulla (permutazione stratificata per profondita')",
         subtitle = paste0(SAMPLE_ID, "  |  blu = nullo (media e 95%, atteso 0.05)  |  rosso = dati reali"),
         y = "Frazione di geni con p < 0.05", x = NULL)
  options(repr.plot.width = 10, repr.plot.height = 6)
  print(p_calib)
  ggsave(file.path(PLOT_DIR, "07_calibrazione_profondita.png"), p_calib, width = 9, height = 6, dpi = 150)
}

## 15. PATHWAY DI CONTESTO E UNIVERSO PER L'ARRICCHIMENTO

In [ ]:
# ##############################################################################
# ##   15. PATHWAY DI CONTESTO E UNIVERSO PER L'ARRICCHIMENTO                 ##
# ##############################################################################
 
sezione_inizio("15. Pathway per il contesto:", ENRICHMENT_CONTEXT, "")
 
PATHWAY_SETS <- list(
  tumor = list(
    hallmark = c("HALLMARK_HYPOXIA", "HALLMARK_GLYCOLYSIS", "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION",
                "HALLMARK_ANGIOGENESIS", "HALLMARK_G2M_CHECKPOINT", "HALLMARK_E2F_TARGETS",
                "HALLMARK_MYC_TARGETS_V1", "HALLMARK_MTORC1_SIGNALING", "HALLMARK_PI3K_AKT_MTOR_SIGNALING",
                "HALLMARK_P53_PATHWAY", "HALLMARK_APOPTOSIS", "HALLMARK_KRAS_SIGNALING_UP",
                "HALLMARK_TNFA_SIGNALING_VIA_NFKB", "HALLMARK_INFLAMMATORY_RESPONSE",
                "HALLMARK_INTERFERON_GAMMA_RESPONSE", "HALLMARK_IL6_JAK_STAT3_SIGNALING", "HALLMARK_COMPLEMENT"),
    bp_regex = paste0("HYPOXIA|ANGIOGEN|VASCUL|CELL_CYCLE|MITOTIC|DNA_REPLICATION|PROLIFERATION|APOPTO|",
                      "CELL_MIGRATION|EXTRACELLULAR_MATRIX|WOUND_HEALING|IMMUNE|INFLAMMAT|CYTOKINE|",
                      "INTERFERON|CHEMOKINE|LEUKOCYTE|MYELOID")
  ),
  inflammation = list(
    hallmark = c("HALLMARK_INFLAMMATORY_RESPONSE", "HALLMARK_TNFA_SIGNALING_VIA_NFKB",
                "HALLMARK_INTERFERON_ALPHA_RESPONSE", "HALLMARK_INTERFERON_GAMMA_RESPONSE",
                "HALLMARK_IL6_JAK_STAT3_SIGNALING", "HALLMARK_COMPLEMENT", "HALLMARK_IL2_STAT5_SIGNALING"),
    bp_regex = paste0("INFLAMMAT|CYTOKINE|INTERLEUKIN|CHEMOKINE|NF_KAPPA|TOLL_LIKE|INTERFERON|",
                      "IMMUNE_RESPONSE|LEUKOCYTE|MICROGLIA")
  )
)
pathway_cfg <- PATHWAY_SETS[[ENRICHMENT_CONTEXT]]
 
get_msig <- function(coll, sub = NULL) {
  a <- names(formals(msigdbr::msigdbr))
  if ("collection" %in% a) {
    msigdbr(species = ORGANISM, collection = coll, subcollection = sub)
  } else {
    msigdbr(species = ORGANISM, category = coll, subcategory = sub)
  }
}
msig_h <- get_msig("H"); msig_bp <- get_msig("C5", "GO:BP")
ens_col <- intersect(c("ensembl_gene", "db_ensembl_gene"), names(msig_h))[1]
 
term2gene <- bind_rows(msig_h %>% select(gs_name, gene = all_of(ens_col)),
                       msig_bp %>% select(gs_name, gene = all_of(ens_col))) %>%
  filter(!is.na(gene)) %>% distinct()
 
all_terms <- unique(term2gene$gs_name)
TARGET_PATHWAYS <- unique(c(intersect(pathway_cfg$hallmark, all_terms),
                           grep(pathway_cfg$bp_regex, grep("^GOBP", all_terms, value = TRUE), value = TRUE)))
cat("Pathway di contesto (", ENRICHMENT_CONTEXT, "):", length(TARGET_PATHWAYS), "su", length(all_terms), "testabili\n")
 
clean_label <- function(id) gsub("_", " ", gsub("^(HALLMARK|GOBP)_", "", id))
 
# --- 15.1  Universo e pathway non ridondanti ----------------------------------
# Universo: geni della tabella unificata. Pathway con 10-500 geni nell'universo.
# Ridondanza: un pathway entra solo se il suo Jaccard con ognuno di quelli gia'
# tenuti e' < 0.5 (prima gli Hallmark, poi i piu' grandi). Ogni GSEA (sezione 16)
# lavora poi solo sui geni testati dal metodo.
U_enr <- sort(unique(as.character(all_results$gene_id)))
stopifnot(all(U_enr %in% rownames(logcounts_raw)))
expr_U <- rowMeans(logcounts_raw)[U_enr]
 
t2g_U <- term2gene %>% filter(gene %in% U_enr)
pn    <- unique(t2g_U$gs_name)
M_pw  <- sparseMatrix(i = match(t2g_U$gs_name, pn), j = match(t2g_U$gene, U_enr), x = 1,
                      dims = c(length(pn), length(U_enr)), dimnames = list(pn, U_enr))
M_pw  <- M_pw[rowSums(M_pw) >= 10 & rowSums(M_pw) <= 500, ]
rs <- rowSums(M_pw); Mt <- t(M_pw); keep <- integer(0)
for (i in order(!grepl("^HALLMARK", rownames(M_pw)), -rs)) {
  v <- as.numeric(M_pw %*% Mt[, i])
  if (length(keep) == 0 || all(v[keep] / (rs[keep] + rs[i] - v[keep]) < 0.5)) keep <- c(keep, i)
}
M_pw <- M_pw[keep, ]
rm(Mt); invisible(gc())
 
T2_pw <- t2g_U %>% filter(gs_name %in% rownames(M_pw))
cat("Pathway non ridondanti:", nrow(M_pw), "| di contesto:", sum(rownames(M_pw) %in% TARGET_PATHWAYS),
    "| geni nell'universo:", length(U_enr), "\n")
stopifnot(sum(rownames(M_pw) %in% TARGET_PATHWAYS) >= 5,
          sum(!rownames(M_pw) %in% TARGET_PATHWAYS) >= 5)

## 16. GSEA SEMPLICE, UNA PER METODO

In [ ]:
# ##############################################################################
# ##   16. GSEA SEMPLICE, UNA PER METODO                                      ##
# ##                                                                          ##
# ##   Una GSEA per metodo sulla classifica completa dei geni TESTATI dal     ##
# ##   metodo (fdr, poi la statistica del test): niente taglio ai primi N,    ##
# ##   niente geni non testati in coda. Punteggio = posizione (N, ..., 1),   ##
# ##   uguale per tutti i metodi (le statistiche non sono confrontabili).     ##
# ##   exponent = 0: conta solo l'ordine dei geni (KS non pesata).             ##
# ##   Tutti i pathway sono testati (pvalueCutoff = 1), BH.                   ##
# ##                                                                          ##
# ##   Per ogni metodo: AUC fra pathway di contesto e altri sul NES, IC       ##
# ##   bootstrap e test di permutazione contro il caso (BH sui metodi). E'    ##
# ##   una misura dentro il metodo; nessun confronto fra metodi.              ##
# ##   Due controlli negativi passano per la stessa GSEA: espressione media   ##
# ##   (Expression_control) e ordine casuale (Random_control).                ##
# ##############################################################################
 
sezione_inizio("16. GSEA semplice per metodo")
 
CONTROLLI <- c("Expression_control", "Random_control")
B_BOOT    <- 1000
ENR_SEED  <- 20261021
 
group_en <- function(g) {
  tr <- function(x) dplyr::recode(x, "Riferimento 2D" = "2D reference",
                                  "Distanza dal centro" = "Distance from centre",
                                  "Tool esterno: SPATA2" = "External tool: SPATA2")
  if (is.factor(g)) factor(tr(as.character(g)), levels = unique(tr(levels(g)))) else tr(as.character(g))
}
 
pos_score <- function(ids) setNames(rev(seq_along(ids)), ids)   # N al primo gene, 1 all'ultimo
 
SCORES <- lapply(setNames(METHOD_ORDER, METHOD_ORDER), function(m) {
  d <- all_results[all_results$Method == m & all_results$tested & !is.na(all_results$rank), ]
  pos_score(d$gene_id[order(d$rank)])
})
set.seed(ENR_SEED)
SCORES$Expression_control <- pos_score(U_enr[order(-expr_U)])
SCORES$Random_control     <- pos_score(sample(U_enr))
 
gsea_all <- bind_rows(lapply(names(SCORES), function(m) {
  stampa("  GSEA: ", m, " (", length(SCORES[[m]]), " geni)")
  set.seed(ENR_SEED)
  g <- as.data.frame(suppressWarnings(GSEA(SCORES[[m]], TERM2GENE = T2_pw, minGSSize = 10, maxGSSize = 500,
                                           pvalueCutoff = 1, pAdjustMethod = "BH", scoreType = "pos", exponent = 0,
                                           verbose = FALSE, seed = TRUE)))
  data.frame(Method = rep(m, nrow(g)), pathway = g$ID, NES = g$NES, pvalue = g$pvalue,
             p_adjust = g$p.adjust, is_context = g$ID %in% TARGET_PATHWAYS, stringsAsFactors = FALSE)
}))
write_csv(gsea_all, file.path(OUTPUT_DIR, "enrichment_gsea_pathways.csv"))
 
enr_tab <- gsea_all %>% group_by(Method) %>%
  summarise(gsea_enriched = sum(p_adjust < ALPHA_FDR), gsea_context = sum(p_adjust < ALPHA_FDR & is_context),
            .groups = "drop") %>%
  mutate(ctx_fraction = ifelse(gsea_enriched > 0, gsea_context / gsea_enriched, NA_real_)) %>%
  left_join(data.frame(Method = names(SCORES), n_tested = lengths(SCORES)), by = "Method") %>%
  left_join(all_results %>% mutate(Method = as.character(Method)) %>% group_by(Method) %>%
              summarise(n_sig = sum(sig), .groups = "drop"), by = "Method") %>%
  mutate(n_sig = ifelse(Method %in% METODI_SENZA_PVALUE, NA_integer_, n_sig))
 
ctx_stats <- function(d) {
  d <- d[!is.na(d$NES), ]; y <- d$is_context; n1 <- sum(y); n0 <- sum(!y)
  if (n1 < 5 || n0 < 5) return(data.frame(simple_n_ctx = n1, simple_auc = NA_real_, simple_lo = NA_real_,
                                          simple_hi = NA_real_, simple_p = NA_real_))
  auc <- function(r, i) (sum(r[i]) - n1 * (n1 + 1) / 2) / (n1 * n0)
  r <- rank(d$NES); a <- auc(r, which(y))
  set.seed(ENR_SEED)
  perm <- replicate(B_BOOT, auc(r, sample.int(length(r), n1)))
  bt   <- replicate(B_BOOT, auc(rank(d$NES[c(sample(which(y), replace = TRUE), sample(which(!y), replace = TRUE))]), seq_len(n1)))
  data.frame(simple_n_ctx = n1, simple_auc = a, simple_lo = unname(quantile(bt, 0.025)),
             simple_hi = unname(quantile(bt, 0.975)), simple_p = (1 + sum(perm >= a)) / (B_BOOT + 1))
}
enr_tab <- enr_tab %>% left_join(
  bind_rows(lapply(split(gsea_all, gsea_all$Method), function(d) cbind(Method = d$Method[1], ctx_stats(d)))) %>%
    mutate(simple_p_adj = p.adjust(simple_p, "BH")), by = "Method")
 
cat("Pathway arricchiti (GSEA, FDR <", ALPHA_FDR, "):\n")
print(enr_tab %>% arrange(desc(gsea_context)) %>%
        select(Method, n_tested, n_sig, gsea_enriched, gsea_context, ctx_fraction, simple_auc, simple_p_adj),
      row.names = FALSE, digits = 3)
 
# --- 16.1  Grafici --------------------------------------------------------------
p_enr3 <- enr_tab %>% filter(!Method %in% CONTROLLI) %>%
  left_join(method_lookup %>% mutate(Method = as.character(Method)), by = "Method") %>%
  mutate(Group = group_en(Group), ctx_fraction = round(ctx_fraction, 2)) %>%
  select(Method, Group, gsea_context, ctx_fraction) %>%
  pivot_longer(c(gsea_context, ctx_fraction), names_to = "Measure", values_to = "n") %>%
  mutate(Measure = recode(Measure, gsea_context = "Context pathways enriched",
                          ctx_fraction = "Fraction of enriched pathways in context"),
         Method = factor(Method, levels = METHOD_ORDER)) %>%
  ggplot(aes(x = Method, y = n, fill = Group)) +
  geom_col() + geom_text(aes(label = n), vjust = -0.3, size = 2.8) +
  facet_grid(Measure ~ Group, scales = "free", space = "free_x") +
  scale_fill_brewer(palette = "Set2") + theme_minimal(base_size = 9) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1), strip.text.y = element_text(angle = 0),
        legend.position = "none") +
  labs(title = paste0("GSEA on the full ranking: ", ENRICHMENT_CONTEXT, " pathways at FDR < ", ALPHA_FDR),
       subtitle = paste0(SAMPLE_ID, " | ", sum(rownames(M_pw) %in% TARGET_PATHWAYS), " context pathways out of ",
                         nrow(M_pw), " | genes tested by each method only"),
       x = NULL, y = NULL)
print(p_enr3)
ggsave(file.path(PLOT_DIR, "04_enrichment_context_counts.png"), p_enr3,
       width = max(10, 0.5 * length(METHOD_ORDER) + 3), height = 6, dpi = 150)
 
gsea_ctx <- gsea_all %>% filter(is_context, p_adjust < ALPHA_FDR, !Method %in% CONTROLLI)
if (nrow(gsea_ctx) > 0) {
  keep_pw <- gsea_ctx %>% count(pathway) %>% arrange(desc(n)) %>% slice_head(n = 40) %>% pull(pathway)
  p_enr4 <- gsea_ctx %>% filter(pathway %in% keep_pw) %>%
    left_join(method_lookup %>% mutate(Method = as.character(Method)), by = "Method") %>%
    mutate(Group = group_en(Group), Label = clean_label(pathway),
           Method = factor(Method, levels = METHOD_ORDER)) %>%
    ggplot(aes(x = Method, y = Label, size = NES, colour = -log10(p_adjust))) +
    geom_point() + scale_colour_viridis_c() +
    facet_grid(~ Group, scales = "free_x", space = "free_x") +
    theme_minimal(base_size = 9) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1), axis.text.y = element_text(size = 7),
          strip.text = element_text(face = "bold")) +
    labs(title = paste0("GSEA: ", ENRICHMENT_CONTEXT, " pathways enriched at FDR < ", ALPHA_FDR),
         subtitle = paste0(SAMPLE_ID, " | up to 40 pathways, most frequent first"),
         x = NULL, y = NULL, colour = "-log10(FDR)", size = "NES")
  print(p_enr4)
  ggsave(file.path(PLOT_DIR, "04b_enrichment_gsea_context.png"), p_enr4,
         width = max(10, 0.5 * length(METHOD_ORDER) + 4), height = max(4, 0.3 * length(keep_pw) + 2), dpi = 150)
}

## 18. SOVRAPPOSIZIONE FRA METODI  (step 22, invariato)

In [ ]:
# ##############################################################################
# ##   18. SOVRAPPOSIZIONE FRA METODI  (step 22, invariato)                   ##
# ##############################################################################
 
sezione_inizio("18. Sovrapposizione fra metodi")
 
sig_sets <- lapply(METHOD_ORDER, function(m) all_results$gene_id[all_results$Method == m & all_results$sig])
names(sig_sets) <- METHOD_ORDER
jac_mat <- outer(sig_sets, sig_sets, Vectorize(function(a, b) jac2(a, b)))
# metodi senza p-value: nessun insieme di significativi, Jaccard non applicabile
senza_p <- rownames(jac_mat) %in% METODI_SENZA_PVALUE
jac_mat[senza_p, ] <- NA
jac_mat[, senza_p] <- NA
write.csv(round(jac_mat, 3), file.path(OUTPUT_DIR, "jaccard_significant_genes.csv"))
 
cat("Jaccard dei geni significativi:\n"); print(round(jac_mat, 3))
 
p_jac <- as.data.frame(as.table(jac_mat)) %>%
  ggplot(aes(factor(Var1, levels = METHOD_ORDER), factor(Var2, levels = rev(METHOD_ORDER)), fill = Freq)) +
  geom_tile(colour = "white") + geom_text(aes(label = ifelse(is.na(Freq), "", sprintf("%.2f", Freq))), size = 3) +
  scale_fill_viridis_c(limits = c(0, 1), na.value = "grey90") + theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = paste0("Jaccard dei geni significativi - ", length(METHOD_ORDER), " metodi"), subtitle = SAMPLE_ID,
       x = NULL, y = NULL, fill = "Jaccard")
print(p_jac)
ggsave(file.path(PLOT_DIR, "05_jaccard_overlap.png"), p_jac,
      width = max(8, 0.6 * length(METHOD_ORDER)), height = max(7, 0.6 * length(METHOD_ORDER)), dpi = 150)

## 19. RIEPILOGO FINALE  (step 23, invariato)

In [ ]:
# ##############################################################################
# ##   19. RIEPILOGO FINALE  (step 23, invariato)                             ##
# ##############################################################################
 
sezione_inizio("19. Riepilogo finale")
 
riepilogo <- all_results %>% group_by(Method) %>%
  summarise(n_significativi = sum(sig), perc_universo = round(100 * mean(sig), 1), .groups = "drop") %>%
  left_join(method_lookup, by = "Method")
 
riepilogo <- riepilogo %>%
  left_join(enr_tab %>% select(Method, simple_auc, simple_lo, simple_hi, gsea_context),
            by = "Method")
if (HAS_GS) riepilogo <- riepilogo %>%
  left_join(data.frame(Method = names(auc_vals), AUC_GS = round(auc_vals, 3)), by = "Method")
if (nrow(auc_ci_tab) > 0) riepilogo <- riepilogo %>%
  left_join(auc_ci_tab %>% transmute(Method, AUC_CI_low = round(CI_low, 3),
                                     AUC_CI_high = round(CI_high, 3)), by = "Method")
 
# Metodi senza p-value (Laplacian_Score): "0 significativi" sarebbe letto come
# un fallimento, mentre quel test non si applica. NA.
riepilogo <- riepilogo %>%
  mutate(across(any_of(c("n_significativi", "perc_universo")),
                ~ ifelse(as.character(Method) %in% METODI_SENZA_PVALUE, NA, .))) %>%
  mutate(Method = factor(Method, levels = METHOD_ORDER)) %>% arrange(Method)
 
cat("\n>>> RIEPILOGO FINALE - ", ENRICHMENT_CONTEXT,
    if (HAS_GS) " - ordine: migliore -> peggiore su Golden Standard" else " - ordine provvisorio (nessun Golden Standard)",
    " <<<\n", sep = "")
print(as.data.frame(riepilogo), row.names = FALSE, digits = 3)
write_csv(riepilogo, file.path(OUTPUT_DIR, "RIEPILOGO_FINALE.csv"))
 
p_final <- riepilogo %>%
  mutate(Group = group_en(Group), simple_auc = round(simple_auc, 3)) %>%
  select(Method, Group, n_significativi, simple_auc) %>%
  pivot_longer(-c(Method, Group), names_to = "Metrica", values_to = "Valore") %>%
  mutate(Metrica = recode(Metrica, n_significativi = "Significant genes",
                          simple_auc = paste0(ENRICHMENT_CONTEXT, " pathway AUC (full ranking)"))) %>%
  ggplot(aes(x = Method, y = Valore, fill = Group)) +
  geom_col() + geom_text(aes(label = Valore), vjust = -0.3, size = 2.8) +
  facet_grid(Metrica ~ Group, scales = "free", space = "free_x") +
  scale_fill_brewer(palette = "Set2") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
        strip.text.y = element_text(angle = 0), legend.position = "none") +
  labs(title = paste0("Quantity vs biological specificity - ", length(METHOD_ORDER), " methods"),
       subtitle = SAMPLE_ID, x = NULL, y = NULL)
print(p_final)
ggsave(file.path(PLOT_DIR, "06_riepilogo_quantita_vs_specificita.png"), p_final,
      width = max(9, 1.6 * n_distinct(riepilogo$Group)), height = 8, dpi = 150)
 
write_csv(all_results, file.path(OUTPUT_DIR, "risultati_tutti_metodi_long.csv"))
 
saveRDS(list(spe = spe, r_center = r_center, center_barcode = CENTER_USED,
             sde_orig = if (HAS_SDE_ORIG) sde_orig else NULL,
             glm_lin = glm_lin, glm_gau = glm_gau,
             glm_pt = glm_pt_results,
             hybrid_glm = if (RUN_HYBRID) hyg else NULL,
             all_results = all_results, riepilogo = riepilogo,
             calibrazione = calib_tab, auc_ci = auc_ci_tab,
             riferimenti_2D = REF2D_INFO, moran_check_spdep = moran_check,
             nullo_spaziale = null_spaziale_tab,
             spata2_qc = spata2_qc,
             arricchimento = list(metodi = enr_tab)),
        file.path(OUTPUT_DIR, "oggetti_v7.rds"))
 
cat("\nCentro usato      :", CENTER_USED, "\n")
cat("Output in         :", normalizePath(OUTPUT_DIR), "\n")
cat("Plot in           :", normalizePath(PLOT_DIR), "\n")

## 20. FILE DI VERIFICA - un unico .txt per dataset da allegare in chat

In [ ]:
# ##############################################################################
# ##   20. FILE DI VERIFICA - un unico .txt per dataset da allegare in chat   ##
# ##############################################################################
 
sezione_inizio("20. File di verifica")
verifica_path <- file.path(OUTPUT_DIR, paste0("verifica_v7_", gsub("[^A-Za-z0-9_]+", "_", SAMPLE_ID), ".txt"))
sink(verifica_path, split = TRUE)
 
cat("======================================================================\n")
cat("VERIFICA pipeline_v7 -", SAMPLE_ID, "-", format(Sys.time()), "\n")
cat("======================================================================\n\n")
 
cat("--- Ambiente ---\n")
cat("N_CORES:", N_CORES, " | RhpcBLASctl:", HAS_BLASCTL,
    " | plyr attaccato:", "package:plyr" %in% search(), "\n")
cat("Checkpoint:", CHECKPOINT_PATH, " scritto il", format(file.info(CHECKPOINT_PATH)$mtime), "\n")
cat("Centro usato:", CENTER_USED, "\n\n")
 
cat("--- Dati ---\n")
cat("Spot:", ncol(spe), " | Geni:", nrow(spe), "\n")
cat("Spot con pseudotime finito: Gaston", sum(is.finite(spe$isodepth)),
    "| SpaceFlow", sum(is.finite(spe$spaceflow_pt)),
    "| Slingshot", sum(is.finite(spe$slingshot_pt)),
    "| Monocle", sum(is.finite(spe$monocle_pt)), "\n")
if (HAS_GS) cat("Golden Standard:", n_gs, "geni (",
                sprintf("%.1f%%", 100 * n_gs / n_distinct(all_results$gene_id)), "dell'universo)\n")
cat("\n")
 
cat("--- Arricchimento di contesto (16) ---\n")
print(as.data.frame(enr_tab), row.names = FALSE, digits = 3)
cat("\n")
 
cat("--- RIEPILOGO_FINALE ---\n")
print(as.data.frame(riepilogo), row.names = FALSE, digits = 3)
cat("\n")
 
cat("--- Calibrazione sotto ipotesi nulla ---\n")
if (nrow(calib_tab) > 0) print(calib_tab, row.names = FALSE) else cat("(saltata)\n")
cat("\n")
 
cat("--- Nullo spaziale dei GLM radiali (10f) ---\n")
if (nrow(null_spaziale_tab) > 0) print(null_spaziale_tab, row.names = FALSE) else cat("(saltato)\n")
cat("\n")
 
cat("--- Bootstrap AUC (95% CI, marginali) ---\n")
if (nrow(auc_ci_tab) > 0) print(as.data.frame(auc_ci_tab), row.names = FALSE, digits = 4) else cat("(nessun Golden Standard)\n")
cat("\n")
 
cat("--- Riferimenti 2D (10d) ---\n")
print(REF2D_INFO, row.names = FALSE)
if (!is.null(moran_check)) {
  cat("Controllo Moran vs spdep::moran.test:\n"); print(moran_check, row.names = FALSE)
} else cat("Controllo spdep non eseguito\n")
cat("SpatialDE_orig, geni non testati:", SDE_ORIG_NON_TESTATI, "\n\n")
cat("SPATA2 rm_zero_infl:", SPATA2_RM_ZERO_INFL, "\n")
 
cat("--- SPATA2 (11b) ---\n")
if (!is.null(spata2_qc)) print(spata2_qc, row.names = FALSE) else cat("(non eseguito)\n")
cat("\n")
 
cat("======================================================================\n")
cat("FINE VERIFICA\n")
cat("======================================================================\n")
sink()
cat("\n>>> File di verifica:", verifica_path, "<<<\n")
 
# --- fine run -------------------------------------------------------------------
stampa("fine sezione ", SEZ_CORRENTE, " (", minuti_da(T_INIZIO_SEZ), " min)")
stampa("RUN COMPLETATA: ", SAMPLE_ID, " in ", minuti_da(T_INIZIO_RUN), " min")